In [1]:
1+1

2

In [2]:
%pip install --upgrade pandas



  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.3
    Uninstalling pandas-3.0.3:
      Successfully uninstalled pandas-3.0.3
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import numpy as np
# Fix pandas import issues caused by a broken/circular install or a local shadowing module
# (for example, a file named pandas.py in the project root).
try:
    import pandas as pd
except Exception as exc:
    raise ImportError(
        "pandas could not be imported. "
        "This is usually caused by a broken pandas installation or a local file named "
        "'pandas.py' shadowing the package. "
        "Reinstall it in the active environment with:\n"
        "python -m pip install --upgrade pandas"
    ) from exc
import h5py

# --------------------------------------------------
# PROJECT ROOT
# --------------------------------------------------

# This assumes the notebook is inside:
# number-simplex-reproduction/notebooks/

ROOT = Path("..").resolve()

DATA_RAW = ROOT / "data" / "raw"
TABLES = ROOT / "tables"
FIGURES = ROOT / "figures"

TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

SUBJECTS = [
    "YFF", "YFI", "YFJ", "YFK", "YFL",
    "YFM", "YFP", "YFR", "YFS", "YFT", "YFU"
]

print("ROOT:", ROOT)
print("RAW DATA:", DATA_RAW)
print("TABLES:", TABLES)
print("FIGURES:", FIGURES)

print("\nSubject folders:")
for subject in SUBJECTS:
    subject_dir = DATA_RAW / subject / "arithmetic"
    print(
        subject,
        "->",
        subject_dir.exists(),
        subject_dir
    )

ROOT: C:\Users\shafi\number-simplex-reproduction
RAW DATA: C:\Users\shafi\number-simplex-reproduction\data\raw
TABLES: C:\Users\shafi\number-simplex-reproduction\tables
FIGURES: C:\Users\shafi\number-simplex-reproduction\figures

Subject folders:
YFF -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFF\arithmetic
YFI -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFI\arithmetic
YFJ -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFJ\arithmetic
YFK -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFK\arithmetic
YFL -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFL\arithmetic
YFM -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFM\arithmetic
YFP -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFP\arithmetic
YFR -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFR\arithmetic
YFS -> True C:\Users\shafi\number-simplex-reproduction\data\raw\YFS\arithmetic
YFT -> True C:\Users\shafi\number-simplex-

In [4]:
rows = []

for subject in SUBJECTS:
    arithmetic_dir = DATA_RAW / subject / "arithmetic"

    csv_file = arithmetic_dir / "photoBehavEvents.csv"
    behav_mat = arithmetic_dir / "photoBehavEvents.mat"
    spikes_mat = arithmetic_dir / "spikes.mat"

    rows.append({
        "subject": subject,
        "csv_exists": csv_file.exists(),
        "behavior_mat_exists": behav_mat.exists(),
        "spikes_exists": spikes_mat.exists(),
    })

file_check = pd.DataFrame(rows)

display(file_check)

file_check.to_csv(
    TABLES / "arithmetic_file_check.csv",
    index=False
)

,subject,csv_exists,behavior_mat_exists,spikes_exists
0,YFF,True,True,True
1,YFI,True,True,True
2,YFJ,True,True,True
3,YFK,True,True,False
4,YFL,True,True,False
5,YFM,True,True,False
6,YFP,True,True,True
7,YFR,True,True,True
8,YFS,True,True,True
9,YFT,True,True,True


In [5]:
subject = "YFF"

behav_path = DATA_RAW / subject / "arithmetic" / "photoBehavEvents.csv"

behav = pd.read_csv(behav_path)

print("Shape:", behav.shape)

display(behav.head())

print("\nColumns:")
for i, col in enumerate(behav.columns):
    print(i, col)

Shape: (100, 22)


,trial,tCue1,tCue2,tCue3,presentationEnd,tPress1,tPress2,tPress3,keyPress1,keyPress2,...,timeFeedback,cue1,cue2,operation,operationFirst,correctAnsw,givenAnsw,correct,correctFb,toExclude
0,1,5338.733333,6089.466667,6840.200000,7607.600000,13830.233333,14030.500000,NaN,1,1.0,...,15398.433333,5,16,-,1,-11,11,0,1,0
1,2,15882.233333,16632.933333,17383.666667,18167.733333,20453.266667,NaN,NaN,3,NaN,...,21938.033333,2,1,+,0,3,3,1,1,0
2,3,22438.533333,23189.233333,23939.966667,24690.666667,26192.133333,NaN,NaN,7,NaN,...,27576.766667,6,1,+,1,7,7,1,1,0
3,4,28077.300000,28828.000000,29578.700000,30329.433333,34967.200000,35250.833333,NaN,2,1.0,...,36635.466667,0,16,+,0,16,21,0,1,0
4,5,37136.000000,37886.700000,38637.400000,39388.133333,40872.900000,41256.600000,NaN,1,6.0,...,42607.900000,10,6,+,0,16,16,1,1,0



Columns:
0 trial
1 tCue1
2 tCue2
3 tCue3
4 presentationEnd
5 tPress1
6 tPress2
7 tPress3
8 keyPress1
9 keyPress2
10 keyPress3
11 choiceEnd
12 timeFeedback
13 cue1
14 cue2
15 operation
16 operationFirst
17 correctAnsw
18 givenAnsw
19 correct
20 correctFb
21 toExclude


In [6]:
print("\nData types:")
print(behav.dtypes)

print("\nMissing values per column:")
print(behav.isna().sum())


Data types:
trial                int64
tCue1              float64
tCue2              float64
tCue3              float64
presentationEnd    float64
tPress1            float64
tPress2            float64
tPress3            float64
keyPress1              str
keyPress2          float64
keyPress3          float64
choiceEnd          float64
timeFeedback       float64
cue1                 int64
cue2                 int64
operation              str
operationFirst       int64
correctAnsw          int64
givenAnsw            int64
correct              int64
correctFb            int64
toExclude            int64
dtype: object

Missing values per column:
trial               0
tCue1               0
tCue2               0
tCue3               0
presentationEnd     0
tPress1             0
tPress2            39
tPress3            96
keyPress1           0
keyPress2          39
keyPress3          96
choiceEnd           0
timeFeedback        0
cue1                0
cue2                0
operation           0

In [7]:
subject = "YFF"

spikes_path = DATA_RAW / subject / "arithmetic" / "spikes.mat"

with h5py.File(spikes_path, "r") as f:
    print("Top-level keys:")
    print(list(f.keys()))

    for key in f.keys():
        obj = f[key]
        print("\nKEY:", key)
        print("TYPE:", type(obj))
        
        if hasattr(obj, "shape"):
            print("SHAPE:", obj.shape)

        print("ATTRIBUTES:")
        for attr_name, attr_value in obj.attrs.items():
            print("   ", attr_name, "=", attr_value)

Top-level keys:
['#refs#', 'regionsVect', 'spikes']

KEY: #refs#
TYPE: <class 'h5py._hl.group.Group'>
ATTRIBUTES:

KEY: regionsVect
TYPE: <class 'h5py._hl.dataset.Dataset'>
SHAPE: (1, 54)
ATTRIBUTES:
    MATLAB_class = b'cell'

KEY: spikes
TYPE: <class 'h5py._hl.group.Group'>
ATTRIBUTES:
    MATLAB_class = b'logical'
    MATLAB_int_decode = 1
    MATLAB_sparse = 54


In [8]:
from scipy import sparse

def load_matlab_sparse_hdf5(filepath, variable="spikes"):
    """
    Load a MATLAB v7.3 sparse matrix stored in HDF5 format.

    Returns
    -------
    scipy.sparse.csc_matrix
    """

    filepath = Path(filepath)

    with h5py.File(filepath, "r") as f:

        if variable not in f:
            raise KeyError(
                f"Variable '{variable}' not found in {filepath}. "
                f"Available keys: {list(f.keys())}"
            )

        g = f[variable]

        required = {"ir", "jc", "data"}

        if not required.issubset(g.keys()):
            raise ValueError(
                f"{variable} does not look like a MATLAB sparse matrix.\n"
                f"Found keys: {list(g.keys())}"
            )

        ir = np.asarray(g["ir"]).ravel().astype(np.int64)
        jc = np.asarray(g["jc"]).ravel().astype(np.int64)
        data = np.asarray(g["data"]).ravel()

        # MATLAB_sparse attribute stores number of rows
        n_rows = int(np.asarray(g.attrs["MATLAB_sparse"]).ravel()[0])

        n_cols = len(jc) - 1

        mat = sparse.csc_matrix(
            (data, ir, jc),
            shape=(n_rows, n_cols)
        )

    return mat

In [9]:
spikes = load_matlab_sparse_hdf5(
    DATA_RAW / "YFF" / "arithmetic" / "spikes.mat"
)

print("Shape:", spikes.shape)
print("Nonzero entries:", spikes.nnz)
print("Sparse format:", spikes.getformat())

Shape: (54, 708276)
Nonzero entries: 501537
Sparse format: csc


In [10]:
def inspect_sparse_matrix(mat):
    """
    Check sparse neural matrix for NaN / Inf / empty rows.
    """

    data = mat.data

    print("Matrix shape:", mat.shape)
    print("Nonzero entries:", mat.nnz)

    print("\nStored values:")
    print("NaN values:", np.isnan(data).sum())
    print("Inf values:", np.isinf(data).sum())

    row_nnz = np.diff(mat.tocsr().indptr)

    empty_rows = np.where(row_nnz == 0)[0]

    print("\nNeurons with zero stored spikes:", len(empty_rows))

    if len(empty_rows) > 0:
        print("Empty neuron rows:", empty_rows)

    return {
        "n_neurons": mat.shape[0],
        "n_timepoints": mat.shape[1],
        "nnz": mat.nnz,
        "nan_values": int(np.isnan(data).sum()),
        "inf_values": int(np.isinf(data).sum()),
        "empty_neurons": len(empty_rows)
    }


diagnostic = inspect_sparse_matrix(spikes)
diagnostic

Matrix shape: (54, 708276)
Nonzero entries: 501537

Stored values:
NaN values: 0
Inf values: 0

Neurons with zero stored spikes: 0


{'n_neurons': 54,
 'n_timepoints': 708276,
 'nnz': 501537,
 'nan_values': 0,
 'inf_values': 0,
 'empty_neurons': 0}

In [11]:
subject = "YFF"

behav_path = DATA_RAW / subject / "arithmetic" / "photoBehavEvents.csv"
behav = pd.read_csv(behav_path)

print("Shape:", behav.shape)

print("\nColumns:")
for i, col in enumerate(behav.columns):
    print(i, col)

print("\nFirst 5 rows:")
display(behav.head())

print("\nMissing values:")
print(behav.isna().sum())

Shape: (100, 22)

Columns:
0 trial
1 tCue1
2 tCue2
3 tCue3
4 presentationEnd
5 tPress1
6 tPress2
7 tPress3
8 keyPress1
9 keyPress2
10 keyPress3
11 choiceEnd
12 timeFeedback
13 cue1
14 cue2
15 operation
16 operationFirst
17 correctAnsw
18 givenAnsw
19 correct
20 correctFb
21 toExclude

First 5 rows:


,trial,tCue1,tCue2,tCue3,presentationEnd,tPress1,tPress2,tPress3,keyPress1,keyPress2,...,timeFeedback,cue1,cue2,operation,operationFirst,correctAnsw,givenAnsw,correct,correctFb,toExclude
0,1,5338.733333,6089.466667,6840.200000,7607.600000,13830.233333,14030.500000,NaN,1,1.0,...,15398.433333,5,16,-,1,-11,11,0,1,0
1,2,15882.233333,16632.933333,17383.666667,18167.733333,20453.266667,NaN,NaN,3,NaN,...,21938.033333,2,1,+,0,3,3,1,1,0
2,3,22438.533333,23189.233333,23939.966667,24690.666667,26192.133333,NaN,NaN,7,NaN,...,27576.766667,6,1,+,1,7,7,1,1,0
3,4,28077.300000,28828.000000,29578.700000,30329.433333,34967.200000,35250.833333,NaN,2,1.0,...,36635.466667,0,16,+,0,16,21,0,1,0
4,5,37136.000000,37886.700000,38637.400000,39388.133333,40872.900000,41256.600000,NaN,1,6.0,...,42607.900000,10,6,+,0,16,16,1,1,0



Missing values:
trial               0
tCue1               0
tCue2               0
tCue3               0
presentationEnd     0
tPress1             0
tPress2            39
tPress3            96
keyPress1           0
keyPress2          39
keyPress3          96
choiceEnd           0
timeFeedback        0
cue1                0
cue2                0
operation           0
operationFirst      0
correctAnsw         0
givenAnsw           0
correct             0
correctFb           0
toExclude           0
dtype: int64


In [12]:
candidate_cols = [
    c for c in behav.columns
    if any(
        key in c.lower()
        for key in [
            "operand",
            "cue",
            "onset",
            "sign",
            "operation",
            "response",
            "correct",
            "rt"
        ]
    )
]

print(candidate_cols)

display(behav[candidate_cols].head(10))

['tCue1', 'tCue2', 'tCue3', 'cue1', 'cue2', 'operation', 'operationFirst', 'correctAnsw', 'correct', 'correctFb']


,tCue1,tCue2,tCue3,cue1,cue2,operation,operationFirst,correctAnsw,correct,correctFb
0,5338.733333,6089.466667,6840.200000,5,16,-,1,-11,0,1
1,15882.233333,16632.933333,17383.666667,2,1,+,0,3,1,1
2,22438.533333,23189.233333,23939.966667,6,1,+,1,7,1,1
3,28077.300000,28828.000000,29578.700000,0,16,+,0,16,0,1
4,37136.000000,37886.700000,38637.400000,10,6,+,0,16,1,1
5,43108.400000,43859.100000,44609.800000,9,8,-,0,1,1,1
6,49247.600000,49998.333333,50749.033333,5,15,-,1,-10,0,1
7,58206.233333,58956.933333,59707.633333,8,1,+,0,9,1,1
8,64245.366667,64996.066667,65746.766667,9,1,+,0,10,1,1
9,70885.066667,71635.766667,72386.500000,0,16,-,0,-16,0,1


In [13]:
def add_operand_onsets(df, subject):
    """
    Add onset times for operand 1 and operand 2.

    Normal subjects:
        operationFirst = 1:
            Cue1 = operation
            Cue2 = operand1
            Cue3 = operand2

        operationFirst = 0:
            Cue1 = operand1
            Cue2 = operand2
            Cue3 = operation

    Special subjects YFR/YFS:
        Cue2 = operation
        Cue1 = operand1
        Cue3 = operand2
    """

    df = df.copy()

    if subject in ["YFR", "YFS"]:
        df["operand1_onset"] = df["tCue1"]
        df["operand2_onset"] = df["tCue3"]

    else:
        op_first = df["operationFirst"].astype(bool)

        df["operand1_onset"] = np.where(
            op_first,
            df["tCue2"],
            df["tCue1"]
        )

        df["operand2_onset"] = np.where(
            op_first,
            df["tCue3"],
            df["tCue2"]
        )

    return df

In [14]:
behav_yff = pd.read_csv(
    DATA_RAW / "YFF" / "arithmetic" / "photoBehavEvents.csv"
)

behav_yff = add_operand_onsets(
    behav_yff,
    subject="YFF"
)

display(
    behav_yff[
        [
            "trial",
            "cue1",
            "cue2",
            "operation",
            "operationFirst",
            "tCue1",
            "tCue2",
            "tCue3",
            "operand1_onset",
            "operand2_onset"
        ]
    ].head(15)
)

,trial,cue1,cue2,operation,operationFirst,tCue1,tCue2,tCue3,operand1_onset,operand2_onset
0,1,5,16,-,1,5338.733333,6089.466667,6840.200000,6089.466667,6840.200000
1,2,2,1,+,0,15882.233333,16632.933333,17383.666667,15882.233333,16632.933333
2,3,6,1,+,1,22438.533333,23189.233333,23939.966667,23189.233333,23939.966667
3,4,0,16,+,0,28077.300000,28828.000000,29578.700000,28077.300000,28828.000000
4,5,10,6,+,0,37136.000000,37886.700000,38637.400000,37136.000000,37886.700000
5,6,9,8,-,0,43108.400000,43859.100000,44609.800000,43108.400000,43859.100000
6,7,5,15,-,1,49247.600000,49998.333333,50749.033333,49998.333333,50749.033333
7,8,8,1,+,0,58206.233333,58956.933333,59707.633333,58206.233333,58956.933333
8,9,9,1,+,0,64245.366667,64996.066667,65746.766667,64245.366667,64996.066667
9,10,0,16,-,0,70885.066667,71635.766667,72386.500000,70885.066667,71635.766667


In [15]:
print(
    behav_yff[
        [
            "operationFirst",
            "tCue1",
            "tCue2",
            "tCue3",
            "operand1_onset",
            "operand2_onset"
        ]
    ].head(20)
)

    operationFirst          tCue1          tCue2          tCue3  \
0                1    5338.733333    6089.466667    6840.200000   
1                0   15882.233333   16632.933333   17383.666667   
2                1   22438.533333   23189.233333   23939.966667   
3                0   28077.300000   28828.000000   29578.700000   
4                0   37136.000000   37886.700000   38637.400000   
5                0   43108.400000   43859.100000   44609.800000   
6                1   49247.600000   49998.333333   50749.033333   
7                0   58206.233333   58956.933333   59707.633333   
8                0   64245.366667   64996.066667   65746.766667   
9                0   70885.066667   71635.766667   72386.500000   
10               1   77391.300000   78142.033333   78892.733333   
11               1   85315.600000   86066.300000   86817.033333   
12               0   90971.033333   91721.733333   92472.466667   
13               1   98361.466667   99112.166667   99862.86666

In [16]:
print("Spike matrix shape:", spikes.shape)
print("Number of time samples:", spikes.shape[1])

print("\nBehavior timing range:")
print("Earliest Cue1:", behav_yff["tCue1"].min())
print("Latest Cue3:", behav_yff["tCue3"].max())
print("Latest presentation end:", behav_yff["presentationEnd"].max())

print("\nExample cue times:")
display(
    behav_yff[
        ["trial", "tCue1", "tCue2", "tCue3",
         "operand1_onset", "operand2_onset"]
    ].head(10)
)

Spike matrix shape: (54, 708276)
Number of time samples: 708276

Behavior timing range:
Earliest Cue1: 5338.73333333333
Latest Cue3: 702575.166666667
Latest presentation end: 703325.866666667

Example cue times:


,trial,tCue1,tCue2,tCue3,operand1_onset,operand2_onset
0,1,5338.733333,6089.466667,6840.200000,6089.466667,6840.200000
1,2,15882.233333,16632.933333,17383.666667,15882.233333,16632.933333
2,3,22438.533333,23189.233333,23939.966667,23189.233333,23939.966667
3,4,28077.300000,28828.000000,29578.700000,28077.300000,28828.000000
4,5,37136.000000,37886.700000,38637.400000,37136.000000,37886.700000
5,6,43108.400000,43859.100000,44609.800000,43108.400000,43859.100000
6,7,49247.600000,49998.333333,50749.033333,49998.333333,50749.033333
7,8,58206.233333,58956.933333,59707.633333,58206.233333,58956.933333
8,9,64245.366667,64996.066667,65746.766667,64245.366667,64996.066667
9,10,70885.066667,71635.766667,72386.500000,70885.066667,71635.766667


In [17]:
WINDOW_START = 0.05
WINDOW_END = 0.95

SAMPLE_RATE = 1000   # spike matrix columns per second

START_OFFSET = int(WINDOW_START * SAMPLE_RATE)
END_OFFSET = int(WINDOW_END * SAMPLE_RATE)

print("Start offset:", START_OFFSET)
print("End offset:", END_OFFSET)
print("Window samples:", END_OFFSET - START_OFFSET)

Start offset: 50
End offset: 950
Window samples: 900


In [18]:
trial_idx = 0
neuron_idx = 0

onset = behav_yff.loc[trial_idx, "operand1_onset"]

onset_sample = int(round(onset * SAMPLE_RATE))

start = onset_sample + START_OFFSET
end = onset_sample + END_OFFSET

print("Trial:", trial_idx)
print("Neuron:", neuron_idx)
print("Operand-1 onset:", onset)
print("Onset sample:", onset_sample)

print("\nAnalysis window:")
print("start =", start)
print("end   =", end)
print("length =", end - start)

print("\nWithin recording:",
      start >= 0 and end <= spikes.shape[1])

Trial: 0
Neuron: 0
Operand-1 onset: 6089.46666666667
Onset sample: 6089467

Analysis window:
start = 6089517
end   = 6090417
length = 900

Within recording: False


In [19]:
response = spikes[neuron_idx, start:end].toarray().ravel()

print("Response shape:", response.shape)
print("NaN:", np.isnan(response).sum())
print("Inf:", np.isinf(response).sum())

print("Sum of values:", response.sum())
print("Nonzero time bins:", np.count_nonzero(response))

Response shape: (0,)
NaN: 0
Inf: 0
Sum of values: 0
Nonzero time bins: 0


In [20]:
rows = []

for trial_idx, row in behav_yff.iterrows():

    onset = row["operand1_onset"]

    if pd.isna(onset):
        rows.append({
            "trial": trial_idx,
            "valid": False,
            "reason": "missing_onset"
        })
        continue

    onset_sample = int(round(onset * SAMPLE_RATE))

    start = onset_sample + START_OFFSET
    end = onset_sample + END_OFFSET

    if start < 0 or end > spikes.shape[1]:
        rows.append({
            "trial": trial_idx,
            "valid": False,
            "reason": "outside_recording"
        })
        continue

    rows.append({
        "trial": trial_idx,
        "valid": True,
        "reason": "",
        "onset": onset,
        "start_sample": start,
        "end_sample": end
    })

trial_check_yff = pd.DataFrame(rows)

print(trial_check_yff["valid"].value_counts(dropna=False))

display(trial_check_yff.head(10))

valid
False    100
Name: count, dtype: int64


,trial,valid,reason
0,0,False,outside_recording
1,1,False,outside_recording
2,2,False,outside_recording
3,3,False,outside_recording
4,4,False,outside_recording
5,5,False,outside_recording
6,6,False,outside_recording
7,7,False,outside_recording
8,8,False,outside_recording
9,9,False,outside_recording


In [21]:
print("Total behavioral trials:", len(behav_yff))
print("Valid operand-1 windows:", trial_check_yff["valid"].sum())
print("Invalid windows:", (~trial_check_yff["valid"]).sum())

if (~trial_check_yff["valid"]).any():
    print("\nReasons:")
    print(
        trial_check_yff.loc[
            ~trial_check_yff["valid"], "reason"
        ].value_counts()
    )

Total behavioral trials: 100
Valid operand-1 windows: 0
Invalid windows: 100

Reasons:
reason
outside_recording    100
Name: count, dtype: int64


In [22]:
print("Spike matrix columns:", spikes.shape[1])

print("\nFirst 10 behavioral timing values:")
display(
    behav_yff[
        [
            "trial",
            "tCue1",
            "tCue2",
            "tCue3",
            "operand1_onset"
        ]
    ].head(10)
)

print("\nTiming ranges:")
for col in [
    "tCue1",
    "tCue2",
    "tCue3",
    "operand1_onset",
    "presentationEnd"
]:
    print(
        f"{col:20s}",
        "min =", behav_yff[col].min(),
        "max =", behav_yff[col].max()
    )

Spike matrix columns: 708276

First 10 behavioral timing values:


,trial,tCue1,tCue2,tCue3,operand1_onset
0,1,5338.733333,6089.466667,6840.200000,6089.466667
1,2,15882.233333,16632.933333,17383.666667,15882.233333
2,3,22438.533333,23189.233333,23939.966667,23189.233333
3,4,28077.300000,28828.000000,29578.700000,28077.300000
4,5,37136.000000,37886.700000,38637.400000,37136.000000
5,6,43108.400000,43859.100000,44609.800000,43108.400000
6,7,49247.600000,49998.333333,50749.033333,49998.333333
7,8,58206.233333,58956.933333,59707.633333,58206.233333
8,9,64245.366667,64996.066667,65746.766667,64245.366667
9,10,70885.066667,71635.766667,72386.500000,70885.066667



Timing ranges:
tCue1                min = 5338.73333333333 max = 701090.433333333
tCue2                min = 6089.46666666667 max = 701824.433333333
tCue3                min = 6840.2 max = 702575.166666667
operand1_onset       min = 6089.46666666667 max = 701090.433333333
presentationEnd      min = 7607.6 max = 703325.866666667


In [23]:
onset = behav_yff.loc[0, "operand1_onset"]

print("First operand onset =", onset)
print("Spike recording length =", spikes.shape[1])

print("\nIf onset is already sample/ms index:")
print("start =", round(onset) + 50)
print("end   =", round(onset) + 950)

print("\nIf onset is seconds × 1000:")
print("start =", round(onset * 1000) + 50)
print("end   =", round(onset * 1000) + 950)

First operand onset = 6089.46666666667
Spike recording length = 708276

If onset is already sample/ms index:
start = 6139
end   = 7039

If onset is seconds × 1000:
start = 6089517
end   = 6090417


In [24]:
WINDOW_START_MS = 50
WINDOW_END_MS = 950

rows = []

for trial_idx, row in behav_yff.iterrows():

    onset = row["operand1_onset"]

    if pd.isna(onset):
        rows.append({
            "trial": trial_idx,
            "valid": False,
            "reason": "missing_onset"
        })
        continue

    # Behavioral timing is already in ms / spike-matrix time scale
    onset_sample = int(round(onset))

    start = onset_sample + WINDOW_START_MS
    end = onset_sample + WINDOW_END_MS

    if start < 0 or end > spikes.shape[1]:
        rows.append({
            "trial": trial_idx,
            "valid": False,
            "reason": "outside_recording"
        })
        continue

    rows.append({
        "trial": trial_idx,
        "valid": True,
        "reason": "",
        "onset": onset,
        "onset_sample": onset_sample,
        "start_sample": start,
        "end_sample": end,
        "window_length": end - start
    })

trial_check_yff = pd.DataFrame(rows)

print("Total behavioral trials:", len(behav_yff))
print("Valid operand-1 windows:", trial_check_yff["valid"].sum())
print("Invalid windows:", (~trial_check_yff["valid"]).sum())

print("\nWindow lengths:")
print(trial_check_yff.loc[
    trial_check_yff["valid"],
    "window_length"
].value_counts())

if (~trial_check_yff["valid"]).any():
    print("\nInvalid reasons:")
    print(
        trial_check_yff.loc[
            ~trial_check_yff["valid"],
            "reason"
        ].value_counts()
    )

display(trial_check_yff.head())

Total behavioral trials: 100
Valid operand-1 windows: 100
Invalid windows: 0

Window lengths:
window_length
900    100
Name: count, dtype: int64


,trial,valid,reason,onset,onset_sample,start_sample,end_sample,window_length
0,0,True,,6089.466667,6089,6139,7039,900
1,1,True,,15882.233333,15882,15932,16832,900
2,2,True,,23189.233333,23189,23239,24139,900
3,3,True,,28077.300000,28077,28127,29027,900
4,4,True,,37136.000000,37136,37186,38086,900


In [25]:
trial_idx = 0
neuron_idx = 0

start = int(trial_check_yff.loc[trial_idx, "start_sample"])
end = int(trial_check_yff.loc[trial_idx, "end_sample"])

response = spikes[
    neuron_idx,
    start:end
].toarray().ravel()

print("Neuron:", neuron_idx)
print("Trial:", trial_idx)

print("Window:", start, "to", end)
print("Window length:", len(response))

print("Total spikes:", response.sum())
print("Nonzero bins:", np.count_nonzero(response))

print("NaN:", np.isnan(response).sum())
print("Inf:", np.isinf(response).sum())

print("\nUnique nonzero values:")
print(np.unique(response[response != 0])[:20])

Neuron: 0
Trial: 0
Window: 6139 to 7039
Window length: 900
Total spikes: 20
Nonzero bins: 20
NaN: 0
Inf: 0

Unique nonzero values:
[1]


In [26]:
spike_counts = np.zeros(
    (spikes.shape[0], len(behav_yff)),
    dtype=float
)

for trial_idx, row in trial_check_yff.iterrows():

    if not row["valid"]:
        spike_counts[:, trial_idx] = np.nan
        continue

    start = int(row["start_sample"])
    end = int(row["end_sample"])

    spike_counts[:, trial_idx] = np.asarray(
        spikes[:, start:end].sum(axis=1)
    ).ravel()


print("Spike-count matrix:", spike_counts.shape)
print("NaNs:", np.isnan(spike_counts).sum())
print("Mean spikes/window:", np.nanmean(spike_counts))
print("Min:", np.nanmin(spike_counts))
print("Max:", np.nanmax(spike_counts))

Spike-count matrix: (54, 100)
NaNs: 0
Mean spikes/window: 11.925185185185185
Min: 0.0
Max: 64.0


In [27]:
print("Unique cue1 values:")
print(np.sort(behav_yff["cue1"].unique()))

print("\nUnique cue2 values:")
print(np.sort(behav_yff["cue2"].unique()))

print("\nUnique operations:")
print(behav_yff["operation"].unique())

Unique cue1 values:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]

Unique cue2 values:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]

Unique operations:
<StringArray>
['-', '+']
Length: 2, dtype: str


In [28]:
print("\ncue1 counts:")
print(
    behav_yff["cue1"]
    .value_counts()
    .sort_index()
)

print("\ncue2 counts:")
print(
    behav_yff["cue2"]
    .value_counts()
    .sort_index()
)


cue1 counts:
cue1
0     10
1      4
2      4
3      5
4      2
5      6
6     10
7      9
8      5
9     10
10     2
11     7
12     4
13     9
14     6
15     2
16     5
Name: count, dtype: int64

cue2 counts:
cue2
0      5
1      6
2     11
3      6
4      5
5      1
6      6
7      8
8      7
9      4
10     4
11     2
12     5
13     8
14     8
15     5
16     9
Name: count, dtype: int64


In [29]:
y_operand1 = behav_yff["cue1"].to_numpy()
y_operand2 = behav_yff["cue2"].to_numpy()

In [30]:
print("Operand 1 labels:", np.unique(y_operand1))
print("Operand 2 labels:", np.unique(y_operand2))

Operand 1 labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
Operand 2 labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]


In [31]:
from collections import Counter

print("Operand 1 trial counts:")
print(Counter(y_operand1))

print("\nMinimum trials in any class:")
print(
    pd.Series(y_operand1)
    .value_counts()
    .min()
)


Operand 1 trial counts:
Counter({np.int64(6): 10, np.int64(0): 10, np.int64(9): 10, np.int64(13): 9, np.int64(7): 9, np.int64(11): 7, np.int64(5): 6, np.int64(14): 6, np.int64(8): 5, np.int64(16): 5, np.int64(3): 5, np.int64(2): 4, np.int64(1): 4, np.int64(12): 4, np.int64(10): 2, np.int64(15): 2, np.int64(4): 2})

Minimum trials in any class:
2


In [32]:
min_class_count = (
    pd.Series(y_operand1)
    .value_counts()
    .min()
)

n_splits = min(10, min_class_count)

print("Feasible CV folds:", n_splits)

Feasible CV folds: 2


In [33]:
cols = [
    "trial",
    "cue1",
    "cue2",
    "operation",
    "operationFirst",
    "tCue1",
    "tCue2",
    "tCue3"
]

display(
    behav_yff[cols].head(30)
)

,trial,cue1,cue2,operation,operationFirst,tCue1,tCue2,tCue3
0,1,5,16,-,1,5338.733333,6089.466667,6840.200000
1,2,2,1,+,0,15882.233333,16632.933333,17383.666667
2,3,6,1,+,1,22438.533333,23189.233333,23939.966667
3,4,0,16,+,0,28077.300000,28828.000000,29578.700000
4,5,10,6,+,0,37136.000000,37886.700000,38637.400000
5,6,9,8,-,0,43108.400000,43859.100000,44609.800000
6,7,5,15,-,1,49247.600000,49998.333333,50749.033333
7,8,8,1,+,0,58206.233333,58956.933333,59707.633333
8,9,9,1,+,0,64245.366667,64996.066667,65746.766667
9,10,0,16,-,0,70885.066667,71635.766667,72386.500000


In [34]:
display(
    behav_yff[
        ["cue1", "cue2", "operation", "operationFirst"]
    ]
    .drop_duplicates()
    .sort_values(["cue1", "cue2"])
    .head(50)
)

,cue1,cue2,operation,operationFirst
48,0,0,-,1
53,0,0,-,0
49,0,3,-,1
15,0,6,-,1
77,0,7,+,1
25,0,9,+,0
31,0,13,+,0
3,0,16,+,0
9,0,16,-,0
82,1,2,+,0


In [35]:
for i, col in enumerate(behav_yff.columns):
    print(i, repr(col))

0 'trial'
1 'tCue1'
2 'tCue2'
3 'tCue3'
4 'presentationEnd'
5 'tPress1'
6 'tPress2'
7 'tPress3'
8 'keyPress1'
9 'keyPress2'
10 'keyPress3'
11 'choiceEnd'
12 'timeFeedback'
13 'cue1'
14 'cue2'
15 'operation'
16 'operationFirst'
17 'correctAnsw'
18 'givenAnsw'
19 'correct'
20 'correctFb'
21 'toExclude'
22 'operand1_onset'
23 'operand2_onset'


In [36]:
behav_yff = behav_yff[
    behav_yff["cue1"].between(1, 9)
]

In [37]:
pooled_rows = []

for trial_idx, row in behav_yff.iterrows():

    # -------------------------
    # Operand 1
    # -------------------------
    operand1 = row["cue1"]

    if 1 <= operand1 <= 9:
        pooled_rows.append({
            "trial_idx": trial_idx,
            "trial": row["trial"],
            "operand_position": 1,
            "number": int(operand1),
            "onset": row["operand1_onset"]
        })

    # -------------------------
    # Operand 2
    # -------------------------
    operand2 = row["cue2"]

    if 1 <= operand2 <= 9:
        pooled_rows.append({
            "trial_idx": trial_idx,
            "trial": row["trial"],
            "operand_position": 2,
            "number": int(operand2),
            "onset": row["operand2_onset"]
        })


pooled_yff = pd.DataFrame(pooled_rows)

print("Original arithmetic trials:", len(behav_yff))
print("Valid pooled operand presentations:", len(pooled_yff))

display(pooled_yff.head(20))

Original arithmetic trials: 55
Valid pooled operand presentations: 82


,trial_idx,trial,operand_position,number,onset
0,0,1,1,5,6089.466667
1,1,2,1,2,15882.233333
2,1,2,2,1,16632.933333
3,2,3,1,6,23189.233333
4,2,3,2,1,23939.966667
5,5,6,1,9,43108.400000
6,5,6,2,8,43859.100000
7,6,7,1,5,49998.333333
8,7,8,1,8,58206.233333
9,7,8,2,1,58956.933333


In [38]:
print("Number classes:")
print(np.sort(pooled_yff["number"].unique()))

print("\nPresentations per number:")
print(
    pooled_yff["number"]
    .value_counts()
    .sort_index()
)

print("\nOperand-position counts:")
print(
    pooled_yff["operand_position"]
    .value_counts()
    .sort_index()
)

Number classes:
[1 2 3 4 5 6 7 8 9]

Presentations per number:
number
1    10
2     9
3     9
4     5
5     7
6    11
7    11
8     9
9    11
Name: count, dtype: int64

Operand-position counts:
operand_position
1    55
2    27
Name: count, dtype: int64


In [39]:
pooled_yff["onset_sample"] = (
    pooled_yff["onset"]
    .round()
    .astype(int)
)

pooled_yff["start_sample"] = (
    pooled_yff["onset_sample"] + 50
)

pooled_yff["end_sample"] = (
    pooled_yff["onset_sample"] + 950
)

pooled_yff["valid_window"] = (
    (pooled_yff["start_sample"] >= 0)
    &
    (pooled_yff["end_sample"] <= spikes.shape[1])
)

print(
    pooled_yff["valid_window"]
    .value_counts()
)

print("\nNumber of pooled samples:",
      len(pooled_yff))

print("Valid samples:",
      pooled_yff["valid_window"].sum())

valid_window
True    82
Name: count, dtype: int64

Number of pooled samples: 82
Valid samples: 82


In [40]:
def make_pooled_temporal_features(
    spikes,
    pooled_table,
    neuron_idx,
    bin_ms
):
    window_length = 900

    if window_length % bin_ms != 0:
        raise ValueError(
            f"{bin_ms} ms does not divide "
            f"{window_length} ms."
        )

    n_bins = window_length // bin_ms

    X = []
    y = []

    for _, row in pooled_table.iterrows():

        if not row["valid_window"]:
            continue

        start = int(row["start_sample"])
        end = int(row["end_sample"])

        response = (
            spikes[neuron_idx, start:end]
            .toarray()
            .ravel()
        )

        if len(response) != window_length:
            continue

        binned = (
            response
            .reshape(n_bins, bin_ms)
            .sum(axis=1)
        )

        X.append(binned)
        y.append(int(row["number"]))

    return (
        np.asarray(X, dtype=float),
        np.asarray(y, dtype=int)
    )

In [41]:
X60, y60 = make_pooled_temporal_features(
    spikes=spikes,
    pooled_table=pooled_yff,
    neuron_idx=0,
    bin_ms=60
)

print("X shape:", X60.shape)
print("y shape:", y60.shape)

print("NaNs in X:", np.isnan(X60).sum())
print("Inf in X:", np.isinf(X60).sum())

print("\nClasses:")
print(np.unique(y60))

print("\nClass counts:")
print(
    pd.Series(y60)
    .value_counts()
    .sort_index()
)

X shape: (82, 15)
y shape: (82,)
NaNs in X: 0
Inf in X: 0

Classes:
[1 2 3 4 5 6 7 8 9]

Class counts:
1    10
2     9
3     9
4     5
5     7
6    11
7    11
8     9
9    11
Name: count, dtype: int64


In [42]:
total_from_temporal = X60.sum(axis=1)

total_direct = []

for _, row in pooled_yff.iterrows():

    start = int(row["start_sample"])
    end = int(row["end_sample"])

    count = spikes[0, start:end].sum()

    total_direct.append(float(count))

total_direct = np.asarray(total_direct)

print(
    "Temporal bins reproduce total spike counts:",
    np.allclose(
        total_from_temporal,
        total_direct
    )
)

print("\nFirst 10:")
print(
    pd.DataFrame({
        "label": y60[:10],
        "temporal_sum": total_from_temporal[:10],
        "direct_sum": total_direct[:10]
    })
)

Temporal bins reproduce total spike counts: True

First 10:
   label  temporal_sum  direct_sum
0      5          20.0        20.0
1      2          20.0        20.0
2      1          14.0        14.0
3      6          13.0        13.0
4      1          18.0        18.0
5      9          18.0        18.0
6      8          11.0        11.0
7      5           9.0         9.0
8      8          20.0        20.0
9      1          12.0        12.0


In [43]:
behav_yff = pd.read_csv(
    DATA_RAW / "YFF" / "arithmetic" / "photoBehavEvents.csv"
)

print("Fresh YFF trials:", len(behav_yff))

Fresh YFF trials: 100


In [44]:
behav_yff = add_operand_onsets(
    behav_yff,
    "YFF"
)

print("After adding onsets:", len(behav_yff))

After adding onsets: 100


In [45]:
pooled_rows = []

for trial_idx, row in behav_yff.iterrows():

    # Operand 1
    if 1 <= row["cue1"] <= 9:
        pooled_rows.append({
            "trial_idx": trial_idx,
            "trial": row["trial"],
            "operand_position": 1,
            "number": int(row["cue1"]),
            "onset": row["operand1_onset"]
        })

    # Operand 2
    if 1 <= row["cue2"] <= 9:
        pooled_rows.append({
            "trial_idx": trial_idx,
            "trial": row["trial"],
            "operand_position": 2,
            "number": int(row["cue2"]),
            "onset": row["operand2_onset"]
        })

pooled_yff = pd.DataFrame(pooled_rows)

print("Original arithmetic trials:", len(behav_yff))
print("Pooled operand presentations:", len(pooled_yff))

print("\nBy operand position:")
print(
    pooled_yff["operand_position"]
    .value_counts()
    .sort_index()
)

print("\nBy number:")
print(
    pooled_yff["number"]
    .value_counts()
    .sort_index()
)

Original arithmetic trials: 100
Pooled operand presentations: 109

By operand position:
operand_position
1    55
2    54
Name: count, dtype: int64

By number:
number
1    10
2    15
3    11
4     7
5     7
6    16
7    17
8    12
9    14
Name: count, dtype: int64


In [46]:
pooled_yff["onset_sample"] = (
    pooled_yff["onset"]
    .round()
    .astype(int)
)

pooled_yff["start_sample"] = (
    pooled_yff["onset_sample"] + 50
)

pooled_yff["end_sample"] = (
    pooled_yff["onset_sample"] + 950
)

pooled_yff["valid_window"] = (
    (pooled_yff["start_sample"] >= 0)
    &
    (pooled_yff["end_sample"] <= spikes.shape[1])
)

print(pooled_yff["valid_window"].value_counts())

valid_window
True    109
Name: count, dtype: int64


In [47]:
class_counts = (
    pooled_yff["number"]
    .value_counts()
    .sort_index()
)

print(class_counts)

min_class_count = class_counts.min()
n_splits = min(10, min_class_count)

print("\nMinimum class count:", min_class_count)
print("CV folds:", n_splits)

number
1    10
2    15
3    11
4     7
5     7
6    16
7    17
8    12
9    14
Name: count, dtype: int64

Minimum class count: 7
CV folds: 7


In [49]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

In [51]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold

scikit-learn version: 1.9.0


In [52]:
import numpy as np
import pandas as pd

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold


def matlab_like_lda_cv(X, y, gamma, random_state=0):
    """
    Approximate MATLAB fitcdiscr(..., 'DiscrimType','linear',
                                'Gamma', gamma,
                                'Prior','uniform')
    using scikit-learn.

    Returns pooled held-out accuracy across CV folds.
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.sort(np.unique(y))
    n_classes = len(classes)

    # Must have all 9 number classes
    if n_classes != 9:
        return np.nan

    # Determine maximum feasible CV folds
    class_counts = pd.Series(y).value_counts()
    n_splits = min(10, int(class_counts.min()))

    if n_splits < 2:
        return np.nan

    # Uniform class priors, as in paper
    priors = np.ones(n_classes) / n_classes

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    all_true = []
    all_pred = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # --------------------------------------
        # Remove zero within-class-variance bins
        # based ONLY on training data
        # --------------------------------------
        within_class_variances = []

        for c in classes:
            Xc = X_train[y_train == c]

            # ddof=1 = sample variance
            v = np.var(Xc, axis=0, ddof=1)

            within_class_variances.append(v)

        within_class_variances = np.vstack(
            within_class_variances
        )

        # Keep a feature if it has nonzero variance
        # in at least one class
        keep = np.any(
            within_class_variances > 0,
            axis=0
        )

        if keep.sum() == 0:
            return np.nan

        X_train_use = X_train[:, keep]
        X_test_use = X_test[:, keep]

        # --------------------------------------
        # MATLAB-like regularized linear LDA
        # --------------------------------------
        lda = LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage=gamma,
            priors=priors
        )

        try:
            lda.fit(X_train_use, y_train)
            pred = lda.predict(X_test_use)

        except Exception:
            return np.nan

        all_true.extend(y_test)
        all_pred.extend(pred)

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)

    accuracy = np.mean(all_true == all_pred)

    return accuracy

In [53]:
BIN_SIZES_MS = [60, 75, 90, 100, 150, 180, 225, 300, 450, 900]
GAMMAS = [0.2, 0.5, 0.8]

In [54]:
results = []

for bin_ms in BIN_SIZES_MS:

    X_bin, y_bin = make_pooled_temporal_features(
        spikes=spikes,
        pooled_table=pooled_yff,
        neuron_idx=0,
        bin_ms=bin_ms
    )

    for gamma in GAMMAS:

        acc = matlab_like_lda_cv(
            X_bin,
            y_bin,
            gamma=gamma,
            random_state=0
        )

        results.append({
            "neuron": 0,
            "bin_ms": bin_ms,
            "gamma": gamma,
            "accuracy": acc,
            "n_features": X_bin.shape[1]
        })

neuron0_grid = pd.DataFrame(results)

display(neuron0_grid)

,neuron,bin_ms,gamma,accuracy,n_features
0,0,60,0.2,0.183486,15
1,0,60,0.5,0.155963,15
2,0,60,0.8,0.137615,15
3,0,75,0.2,0.137615,12
4,0,75,0.5,0.119266,12
5,0,75,0.8,0.119266,12
6,0,90,0.2,0.201835,10
7,0,90,0.5,0.192661,10
8,0,90,0.8,0.192661,10
9,0,100,0.2,0.155963,9


In [55]:
best_row = neuron0_grid.loc[
    neuron0_grid["accuracy"].idxmax()
]

print("Best bin size:", best_row["bin_ms"])
print("Best gamma:", best_row["gamma"])
print("Best accuracy:", best_row["accuracy"])
print("Number of temporal features:", best_row["n_features"])

Best bin size: 90.0
Best gamma: 0.2
Best accuracy: 0.2018348623853211
Number of temporal features: 10.0


In [56]:
# Rebuild features using the selected bin size
X_best, y_best = make_pooled_temporal_features(
    spikes=spikes,
    pooled_table=pooled_yff,
    neuron_idx=0,
    bin_ms=90
)

observed_accuracy = matlab_like_lda_cv(
    X_best,
    y_best,
    gamma=0.2,
    random_state=0
)

print("Observed accuracy:", observed_accuracy)
print("X shape:", X_best.shape)

Observed accuracy: 0.2018348623853211
X shape: (109, 10)


In [57]:
N_PERMUTATIONS = 200

rng = np.random.default_rng(0)

permuted_accuracies = []

for p in range(N_PERMUTATIONS):

    y_shuffled = rng.permutation(y_best)

    perm_acc = matlab_like_lda_cv(
        X_best,
        y_shuffled,
        gamma=0.2,
        random_state=0
    )

    permuted_accuracies.append(perm_acc)

permuted_accuracies = np.asarray(
    permuted_accuracies,
    dtype=float
)

print("Permutations:", len(permuted_accuracies))
print("NaNs:", np.isnan(permuted_accuracies).sum())
print("Mean null accuracy:", np.nanmean(permuted_accuracies))
print("Observed accuracy:", observed_accuracy)

Permutations: 200
NaNs: 0
Mean null accuracy: 0.11252293577981652
Observed accuracy: 0.2018348623853211


In [58]:
n_equal_or_better = np.sum(
    permuted_accuracies >= observed_accuracy
)

p_value = (
    1 + n_equal_or_better
) / (
    1 + N_PERMUTATIONS
)

print("Permutations >= observed:", n_equal_or_better)
print("Permutation p-value:", p_value)
print("Coding neuron:", p_value < 0.05)

Permutations >= observed: 4
Permutation p-value: 0.024875621890547265
Coding neuron: True


In [59]:
def analyze_one_temporal_neuron(
    spikes,
    pooled_table,
    neuron_idx,
    bin_sizes,
    gammas,
    n_permutations=200,
    random_state=0
):
    rng = np.random.default_rng(random_state)

    grid_rows = []

    # --------------------------------------------------
    # 1. Search all bin-size / Gamma combinations
    # --------------------------------------------------
    for bin_ms in bin_sizes:

        X_bin, y_bin = make_pooled_temporal_features(
            spikes=spikes,
            pooled_table=pooled_table,
            neuron_idx=neuron_idx,
            bin_ms=bin_ms
        )

        for gamma in gammas:

            acc = matlab_like_lda_cv(
                X_bin,
                y_bin,
                gamma=gamma,
                random_state=random_state
            )

            grid_rows.append({
                "bin_ms": bin_ms,
                "gamma": gamma,
                "accuracy": acc
            })

    grid = pd.DataFrame(grid_rows)

    # --------------------------------------------------
    # 2. Handle neuron if every parameter combination fails
    # --------------------------------------------------
    if grid["accuracy"].isna().all():
        return {
            "neuron": neuron_idx,
            "best_bin_ms": np.nan,
            "best_gamma": np.nan,
            "temporal_accuracy": np.nan,
            "permutation_p": np.nan,
            "coding": False,
            "valid": False,
            "invalid_reason": "all parameter combinations returned NaN"
        }

    # --------------------------------------------------
    # 3. Select best observed parameters
    # --------------------------------------------------
    best_idx = grid["accuracy"].idxmax()
    best = grid.loc[best_idx]

    best_bin = int(best["bin_ms"])
    best_gamma = float(best["gamma"])
    observed_accuracy = float(best["accuracy"])

    # --------------------------------------------------
    # 4. Rebuild features using selected bin
    # --------------------------------------------------
    X_best, y_best = make_pooled_temporal_features(
        spikes=spikes,
        pooled_table=pooled_table,
        neuron_idx=neuron_idx,
        bin_ms=best_bin
    )

    # --------------------------------------------------
    # 5. Permutation test
    # --------------------------------------------------
    permuted_accuracies = []

    for _ in range(n_permutations):

        y_shuffle = rng.permutation(y_best)

        perm_acc = matlab_like_lda_cv(
            X_best,
            y_shuffle,
            gamma=best_gamma,
            random_state=random_state
        )

        permuted_accuracies.append(perm_acc)

    permuted_accuracies = np.asarray(
        permuted_accuracies,
        dtype=float
    )

    # If any permutation failed, flag it rather than hiding it
    if np.isnan(permuted_accuracies).any():
        return {
            "neuron": neuron_idx,
            "best_bin_ms": best_bin,
            "best_gamma": best_gamma,
            "temporal_accuracy": observed_accuracy,
            "permutation_p": np.nan,
            "coding": False,
            "valid": False,
            "invalid_reason": "NaN occurred during permutation test"
        }

    n_equal_or_better = np.sum(
        permuted_accuracies >= observed_accuracy
    )

    p_value = (
        1 + n_equal_or_better
    ) / (
        1 + n_permutations
    )

    return {
        "neuron": neuron_idx,
        "best_bin_ms": best_bin,
        "best_gamma": best_gamma,
        "temporal_accuracy": observed_accuracy,
        "permutation_p": p_value,
        "coding": p_value < 0.05,
        "valid": True,
        "invalid_reason": ""
    }

In [60]:
yff_results = []

for neuron_idx in range(spikes.shape[0]):

    print(
        f"Analyzing neuron "
        f"{neuron_idx + 1}/{spikes.shape[0]}..."
    )

    result = analyze_one_temporal_neuron(
        spikes=spikes,
        pooled_table=pooled_yff,
        neuron_idx=neuron_idx,
        bin_sizes=BIN_SIZES_MS,
        gammas=GAMMAS,
        n_permutations=200,
        random_state=0
    )

    yff_results.append(result)

yff_temporal = pd.DataFrame(yff_results)

Analyzing neuron 1/54...
Analyzing neuron 2/54...
Analyzing neuron 3/54...
Analyzing neuron 4/54...
Analyzing neuron 5/54...
Analyzing neuron 6/54...
Analyzing neuron 7/54...
Analyzing neuron 8/54...
Analyzing neuron 9/54...
Analyzing neuron 10/54...
Analyzing neuron 11/54...
Analyzing neuron 12/54...
Analyzing neuron 13/54...
Analyzing neuron 14/54...
Analyzing neuron 15/54...
Analyzing neuron 16/54...
Analyzing neuron 17/54...
Analyzing neuron 18/54...
Analyzing neuron 19/54...
Analyzing neuron 20/54...
Analyzing neuron 21/54...
Analyzing neuron 22/54...
Analyzing neuron 23/54...
Analyzing neuron 24/54...
Analyzing neuron 25/54...
Analyzing neuron 26/54...
Analyzing neuron 27/54...
Analyzing neuron 28/54...
Analyzing neuron 29/54...
Analyzing neuron 30/54...
Analyzing neuron 31/54...
Analyzing neuron 32/54...
Analyzing neuron 33/54...
Analyzing neuron 34/54...
Analyzing neuron 35/54...
Analyzing neuron 36/54...
Analyzing neuron 37/54...
Analyzing neuron 38/54...
Analyzing neuron 39/5

In [61]:
display(yff_temporal)

,neuron,best_bin_ms,best_gamma,temporal_accuracy,permutation_p,coding,valid,invalid_reason
0,0,90,0.2,0.201835,0.024876,True,True,
1,1,180,0.5,0.211009,0.014925,True,True,
2,2,225,0.8,0.146789,0.149254,False,True,
3,3,900,0.2,0.192661,0.004975,True,True,
4,4,100,0.2,0.201835,0.019900,True,True,
5,5,450,0.5,0.165138,0.084577,False,True,
6,6,225,0.2,0.201835,0.009950,True,True,
7,7,300,0.5,0.155963,0.099502,False,True,
8,8,60,0.8,0.155963,0.179104,False,True,
9,9,90,0.2,0.183486,0.049751,True,True,


In [62]:
print("Total neurons:", len(yff_temporal))

print(
    "Valid neurons:",
    yff_temporal["valid"].sum()
)

print(
    "Invalid neurons:",
    (~yff_temporal["valid"]).sum()
)

print(
    "Temporal coding neurons:",
    yff_temporal["coding"].sum()
)

coding_prop = (
    yff_temporal["coding"].sum()
    / yff_temporal["valid"].sum()
)

print(
    "Temporal coding proportion:",
    coding_prop
)

print(
    "Temporal coding percentage:",
    100 * coding_prop
)

Total neurons: 54
Valid neurons: 54
Invalid neurons: 0
Temporal coding neurons: 16
Temporal coding proportion: 0.2962962962962963
Temporal coding percentage: 29.629629629629626


In [63]:
print("\nNaNs by column:")
print(yff_temporal.isna().sum())

print("\nInvalid reasons:")
print(
    yff_temporal.loc[
        ~yff_temporal["valid"],
        "invalid_reason"
    ].value_counts()
)


NaNs by column:
neuron               0
best_bin_ms          0
best_gamma           0
temporal_accuracy    0
permutation_p        0
coding               0
valid                0
invalid_reason       0
dtype: int64

Invalid reasons:
Series([], Name: count, dtype: int64)


In [64]:
print("Best bin-size counts:")
print(
    yff_temporal["best_bin_ms"]
    .value_counts()
    .sort_index()
)

print("\nBest Gamma counts:")
print(
    yff_temporal["best_gamma"]
    .value_counts()
    .sort_index()
)

print("\nPermutation p-value summary:")
print(
    yff_temporal["permutation_p"].describe()
)

print("\nCoding neurons:")
display(
    yff_temporal.loc[
        yff_temporal["coding"],
        [
            "neuron",
            "best_bin_ms",
            "best_gamma",
            "temporal_accuracy",
            "permutation_p"
        ]
    ].sort_values("permutation_p")
)

Best bin-size counts:
best_bin_ms
60     7
75     8
90     7
100    3
150    6
180    7
225    6
300    3
450    2
900    5
Name: count, dtype: int64

Best Gamma counts:
best_gamma
0.2    27
0.5    12
0.8    15
Name: count, dtype: int64

Permutation p-value summary:
count    54.000000
mean      0.193016
std       0.204398
min       0.004975
25%       0.041045
50%       0.121891
75%       0.261194
max       0.975124
Name: permutation_p, dtype: float64

Coding neurons:


,neuron,best_bin_ms,best_gamma,temporal_accuracy,permutation_p
3,3,900,0.2,0.192661,0.004975
50,50,90,0.2,0.211009,0.004975
32,32,300,0.2,0.211009,0.004975
17,17,60,0.5,0.201835,0.004975
39,39,150,0.5,0.229358,0.009950
6,6,225,0.2,0.201835,0.009950
1,1,180,0.5,0.211009,0.014925
4,4,100,0.2,0.201835,0.019900
34,34,180,0.5,0.220183,0.024876
30,30,75,0.8,0.183486,0.024876


In [65]:
def matlab_gamma_lda_cv(X, y, gamma, random_state=0):

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.sort(np.unique(y))

    if len(classes) != 9:
        return np.nan

    class_counts = pd.Series(y).value_counts()
    n_splits = min(10, int(class_counts.min()))

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    all_true = []
    all_pred = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # ---------------------------------------
        # remove features with zero within-class variance
        # ---------------------------------------
        variances = []

        for c in classes:
            Xc = X_train[y_train == c]
            variances.append(
                np.var(Xc, axis=0, ddof=1)
            )

        variances = np.vstack(variances)

        keep = np.any(variances > 0, axis=0)

        if keep.sum() == 0:
            return np.nan

        X_train = X_train[:, keep]
        X_test = X_test[:, keep]

        # ---------------------------------------
        # class means
        # ---------------------------------------
        means = {}

        for c in classes:
            means[c] = X_train[y_train == c].mean(axis=0)

        # ---------------------------------------
        # pooled within-class covariance
        # ---------------------------------------
        p = X_train.shape[1]

        scatter = np.zeros((p, p))
        total_df = 0

        for c in classes:

            Xc = X_train[y_train == c]

            centered = Xc - means[c]

            scatter += centered.T @ centered

            total_df += len(Xc) - 1

        Sigma = scatter / total_df

        # ---------------------------------------
        # MATLAB Gamma regularization
        #
        # Sigma_gamma =
        # (1-gamma) Sigma
        # + gamma diag(Sigma)
        # ---------------------------------------
        Sigma_gamma = (
            (1 - gamma) * Sigma
            + gamma * np.diag(np.diag(Sigma))
        )

        # numerical-safe inverse
        Sigma_inv = np.linalg.pinv(Sigma_gamma)

        # ---------------------------------------
        # uniform priors
        # ---------------------------------------
        prior = 1.0 / len(classes)

        predictions = []

        for x in X_test:

            scores = []

            for c in classes:

                mu = means[c]

                score = (
                    x @ Sigma_inv @ mu
                    - 0.5 * mu @ Sigma_inv @ mu
                    + np.log(prior)
                )

                scores.append(score)

            predictions.append(
                classes[np.argmax(scores)]
            )

        all_true.extend(y_test)
        all_pred.extend(predictions)

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)

    return np.mean(all_true == all_pred)

In [66]:
results_matlab = []

for bin_ms in BIN_SIZES_MS:

    X_bin, y_bin = make_pooled_temporal_features(
        spikes=spikes,
        pooled_table=pooled_yff,
        neuron_idx=0,
        bin_ms=bin_ms
    )

    for gamma in GAMMAS:

        acc = matlab_gamma_lda_cv(
            X_bin,
            y_bin,
            gamma=gamma,
            random_state=0
        )

        results_matlab.append({
            "bin_ms": bin_ms,
            "gamma": gamma,
            "accuracy": acc
        })

matlab_grid_neuron0 = pd.DataFrame(results_matlab)

display(matlab_grid_neuron0)

,bin_ms,gamma,accuracy
0,60,0.2,0.183486
1,60,0.5,0.174312
2,60,0.8,0.128440
3,75,0.2,0.146789
4,75,0.5,0.128440
5,75,0.8,0.128440
6,90,0.2,0.211009
7,90,0.5,0.211009
8,90,0.8,0.174312
9,100,0.2,0.155963


In [67]:
best_matlab = matlab_grid_neuron0.loc[
    matlab_grid_neuron0["accuracy"].idxmax()
]

print("Best bin:", best_matlab["bin_ms"])
print("Best gamma:", best_matlab["gamma"])
print("Best accuracy:", best_matlab["accuracy"])

Best bin: 90.0
Best gamma: 0.2
Best accuracy: 0.21100917431192662


In [68]:
X_best, y_best = make_pooled_temporal_features(
    spikes=spikes,
    pooled_table=pooled_yff,
    neuron_idx=0,
    bin_ms=90
)

observed_accuracy = matlab_gamma_lda_cv(
    X_best,
    y_best,
    gamma=0.2,
    random_state=0
)

N_PERMUTATIONS = 200
rng = np.random.default_rng(0)

permuted_accuracies = []

for p in range(N_PERMUTATIONS):

    y_shuffled = rng.permutation(y_best)

    perm_acc = matlab_gamma_lda_cv(
        X_best,
        y_shuffled,
        gamma=0.2,
        random_state=0
    )

    permuted_accuracies.append(perm_acc)

permuted_accuracies = np.asarray(permuted_accuracies)

n_equal_or_better = np.sum(
    permuted_accuracies >= observed_accuracy
)

p_value = (
    1 + n_equal_or_better
) / (
    N_PERMUTATIONS + 1
)

print("Observed accuracy:", observed_accuracy)
print("Mean null accuracy:", permuted_accuracies.mean())
print("Permutations >= observed:", n_equal_or_better)
print("Permutation p-value:", p_value)
print("Coding neuron:", p_value < 0.05)

Observed accuracy: 0.21100917431192662
Mean null accuracy: 0.11224770642201837
Permutations >= observed: 3
Permutation p-value: 0.01990049751243781
Coding neuron: True


In [69]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

In [70]:
def matlab_gamma_lda_cv(X, y, gamma, random_state=0):

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.sort(np.unique(y))

    if len(classes) != 9:
        return np.nan

    class_counts = pd.Series(y).value_counts()

    n_splits = min(
        10,
        int(class_counts.min())
    )

    if n_splits < 2:
        return np.nan

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    all_true = []
    all_pred = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # ------------------------------------------------
        # Remove features with zero within-class variance
        # using training data only
        # ------------------------------------------------
        within_class_variances = []

        for c in classes:

            Xc = X_train[y_train == c]

            v = np.var(
                Xc,
                axis=0,
                ddof=1
            )

            within_class_variances.append(v)

        within_class_variances = np.vstack(
            within_class_variances
        )

        keep = np.any(
            within_class_variances > 0,
            axis=0
        )

        if keep.sum() == 0:
            return np.nan

        X_train = X_train[:, keep]
        X_test = X_test[:, keep]

        # ------------------------------------------------
        # Class means
        # ------------------------------------------------
        means = {}

        for c in classes:

            means[c] = X_train[
                y_train == c
            ].mean(axis=0)

        # ------------------------------------------------
        # Pooled within-class covariance
        # ------------------------------------------------
        n_features = X_train.shape[1]

        scatter = np.zeros(
            (n_features, n_features),
            dtype=float
        )

        total_df = 0

        for c in classes:

            Xc = X_train[y_train == c]

            centered = (
                Xc - means[c]
            )

            scatter += (
                centered.T @ centered
            )

            total_df += len(Xc) - 1

        if total_df <= 0:
            return np.nan

        Sigma = scatter / total_df

        # ------------------------------------------------
        # MATLAB Gamma-style regularization
        #
        # Sigma_gamma =
        # (1-gamma)*Sigma +
        # gamma*diag(diag(Sigma))
        # ------------------------------------------------
        Sigma_gamma = (
            (1 - gamma) * Sigma
            + gamma
            * np.diag(
                np.diag(Sigma)
            )
        )

        # Numerical-safe inverse
        Sigma_inv = np.linalg.pinv(
            Sigma_gamma
        )

        # ------------------------------------------------
        # Uniform prior
        # ------------------------------------------------
        prior = 1.0 / len(classes)

        predictions = []

        for x in X_test:

            scores = []

            for c in classes:

                mu = means[c]

                score = (
                    x @ Sigma_inv @ mu
                    - 0.5
                    * mu
                    @ Sigma_inv
                    @ mu
                    + np.log(prior)
                )

                scores.append(score)

            predicted_class = classes[
                np.argmax(scores)
            ]

            predictions.append(
                predicted_class
            )

        all_true.extend(y_test)
        all_pred.extend(predictions)

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)

    return np.mean(
        all_true == all_pred
    )

In [71]:
def analyze_one_temporal_neuron(
    spikes,
    pooled_table,
    neuron_idx,
    bin_sizes,
    gammas,
    n_permutations=200,
    random_state=0
):

    # ====================================================
    # 1. Hyperparameter search
    # ====================================================

    grid_rows = []

    for bin_ms in bin_sizes:

        X_bin, y_bin = make_pooled_temporal_features(
            spikes=spikes,
            pooled_table=pooled_table,
            neuron_idx=neuron_idx,
            bin_ms=bin_ms
        )

        for gamma in gammas:

            try:

                acc = matlab_gamma_lda_cv(
                    X_bin,
                    y_bin,
                    gamma=gamma,
                    random_state=random_state
                )

            except Exception:

                acc = np.nan

            grid_rows.append({
                "bin_ms": bin_ms,
                "gamma": gamma,
                "accuracy": acc
            })

    grid = pd.DataFrame(grid_rows)

    # ====================================================
    # 2. Check whether neuron was analyzable
    # ====================================================

    if grid["accuracy"].isna().all():

        return {
            "neuron": neuron_idx,
            "best_bin_ms": np.nan,
            "best_gamma": np.nan,
            "temporal_accuracy": np.nan,
            "permutation_p": np.nan,
            "n_perm_equal_or_better": np.nan,
            "coding": False,
            "valid": False,
            "invalid_reason":
                "all parameter combinations returned NaN"
        }

    # ====================================================
    # 3. Choose best bin size + Gamma
    # ====================================================

    best_idx = grid["accuracy"].idxmax()

    best = grid.loc[best_idx]

    best_bin = int(
        best["bin_ms"]
    )

    best_gamma = float(
        best["gamma"]
    )

    observed_accuracy = float(
        best["accuracy"]
    )

    # ====================================================
    # 4. Rebuild data at selected bin
    # ====================================================

    X_best, y_best = make_pooled_temporal_features(
        spikes=spikes,
        pooled_table=pooled_table,
        neuron_idx=neuron_idx,
        bin_ms=best_bin
    )

    # ====================================================
    # 5. Permutation test
    # ====================================================

    rng = np.random.default_rng(
        random_state
    )

    permuted_accuracies = []

    for permutation_idx in range(
        n_permutations
    ):

        y_shuffled = rng.permutation(
            y_best
        )

        try:

            perm_acc = matlab_gamma_lda_cv(
                X_best,
                y_shuffled,
                gamma=best_gamma,
                random_state=random_state
            )

        except Exception:

            perm_acc = np.nan

        permuted_accuracies.append(
            perm_acc
        )

    permuted_accuracies = np.asarray(
        permuted_accuracies,
        dtype=float
    )

    # ====================================================
    # 6. Check permutation failures
    # ====================================================

    n_nan_perm = np.isnan(
        permuted_accuracies
    ).sum()

    if n_nan_perm > 0:

        return {
            "neuron": neuron_idx,
            "best_bin_ms": best_bin,
            "best_gamma": best_gamma,
            "temporal_accuracy":
                observed_accuracy,
            "permutation_p": np.nan,
            "n_perm_equal_or_better":
                np.nan,
            "coding": False,
            "valid": False,
            "invalid_reason":
                f"{n_nan_perm} permutation accuracies were NaN"
        }

    # ====================================================
    # 7. One-sided permutation p-value
    # ====================================================

    n_equal_or_better = np.sum(
        permuted_accuracies
        >= observed_accuracy
    )

    p_value = (
        1 + n_equal_or_better
    ) / (
        1 + n_permutations
    )

    # ====================================================
    # 8. Final result
    # ====================================================

    return {
        "neuron": neuron_idx,
        "best_bin_ms": best_bin,
        "best_gamma": best_gamma,
        "temporal_accuracy":
            observed_accuracy,
        "permutation_p":
            float(p_value),
        "n_perm_equal_or_better":
            int(n_equal_or_better),
        "coding":
            bool(p_value < 0.05),
        "valid": True,
        "invalid_reason": ""
    }

In [72]:
BIN_SIZES_MS = [
    60, 75, 90, 100, 150,
    180, 225, 300, 450, 900
]

GAMMAS = [
    0.2,
    0.5,
    0.8
]

N_PERMUTATIONS = 200

In [73]:
yff_results_matlab = []

for neuron_idx in range(
    spikes.shape[0]
):

    print(
        f"Analyzing YFF neuron "
        f"{neuron_idx + 1}/"
        f"{spikes.shape[0]}"
    )

    result = analyze_one_temporal_neuron(
        spikes=spikes,
        pooled_table=pooled_yff,
        neuron_idx=neuron_idx,
        bin_sizes=BIN_SIZES_MS,
        gammas=GAMMAS,
        n_permutations=N_PERMUTATIONS,
        random_state=0
    )

    yff_results_matlab.append(
        result
    )

yff_temporal_matlab = pd.DataFrame(
    yff_results_matlab
)

Analyzing YFF neuron 1/54
Analyzing YFF neuron 2/54
Analyzing YFF neuron 3/54
Analyzing YFF neuron 4/54
Analyzing YFF neuron 5/54
Analyzing YFF neuron 6/54
Analyzing YFF neuron 7/54
Analyzing YFF neuron 8/54
Analyzing YFF neuron 9/54
Analyzing YFF neuron 10/54
Analyzing YFF neuron 11/54
Analyzing YFF neuron 12/54
Analyzing YFF neuron 13/54
Analyzing YFF neuron 14/54
Analyzing YFF neuron 15/54
Analyzing YFF neuron 16/54
Analyzing YFF neuron 17/54
Analyzing YFF neuron 18/54
Analyzing YFF neuron 19/54
Analyzing YFF neuron 20/54
Analyzing YFF neuron 21/54
Analyzing YFF neuron 22/54
Analyzing YFF neuron 23/54
Analyzing YFF neuron 24/54
Analyzing YFF neuron 25/54
Analyzing YFF neuron 26/54
Analyzing YFF neuron 27/54
Analyzing YFF neuron 28/54
Analyzing YFF neuron 29/54
Analyzing YFF neuron 30/54
Analyzing YFF neuron 31/54
Analyzing YFF neuron 32/54
Analyzing YFF neuron 33/54
Analyzing YFF neuron 34/54
Analyzing YFF neuron 35/54
Analyzing YFF neuron 36/54
Analyzing YFF neuron 37/54
Analyzing 

In [74]:
print(
    "Total neurons:",
    len(yff_temporal_matlab)
)

print(
    "Valid neurons:",
    yff_temporal_matlab[
        "valid"
    ].sum()
)

print(
    "Invalid neurons:",
    (~yff_temporal_matlab[
        "valid"
    ]).sum()
)

n_coding = yff_temporal_matlab[
    "coding"
].sum()

n_valid = yff_temporal_matlab[
    "valid"
].sum()

print(
    "Temporal coding neurons:",
    n_coding
)

print(
    "Temporal coding percentage:",
    100 * n_coding / n_valid
)

print("\nNaNs:")
print(
    yff_temporal_matlab
    .isna()
    .sum()
)

Total neurons: 54
Valid neurons: 54
Invalid neurons: 0
Temporal coding neurons: 17
Temporal coding percentage: 31.48148148148148

NaNs:
neuron                    0
best_bin_ms               0
best_gamma                0
temporal_accuracy         0
permutation_p             0
n_perm_equal_or_better    0
coding                    0
valid                     0
invalid_reason            0
dtype: int64


fixing the covariance regularization

In [75]:
yff_temporal_matlab.to_csv(
    TABLES / "YFF_temporal_decoding_results.csv",
    index=False
)

In [76]:
display(
    yff_temporal_matlab.loc[
        yff_temporal_matlab["coding"],
        [
            "neuron",
            "best_bin_ms",
            "best_gamma",
            "temporal_accuracy",
            "permutation_p",
            "n_perm_equal_or_better"
        ]
    ].sort_values("permutation_p")
)

,neuron,best_bin_ms,best_gamma,temporal_accuracy,permutation_p,n_perm_equal_or_better
3,3,900,0.2,0.192661,0.004975,0
6,6,225,0.2,0.211009,0.004975,0
30,30,100,0.8,0.192661,0.004975,0
32,32,300,0.2,0.220183,0.004975,0
33,33,60,0.2,0.201835,0.004975,0
39,39,150,0.5,0.220183,0.009950,1
50,50,90,0.2,0.192661,0.009950,1
4,4,100,0.2,0.201835,0.014925,2
1,1,180,0.8,0.201835,0.019900,3
28,28,60,0.2,0.192661,0.019900,3


In [80]:
def prepare_arithmetic_subject(subject, spikes):

    # -----------------------------
    # Load behavior
    # -----------------------------
    behav_path = (
        DATA_RAW
        / subject
        / "arithmetic"
        / "photoBehavEvents.csv"
    )

    behav = pd.read_csv(behav_path)

    # -----------------------------
    # Operand onset mapping
    # -----------------------------
    behav = add_operand_onsets(
        behav,
        subject
    )

    # -----------------------------
    # Pool operand 1 + operand 2
    # Keep only number classes 1–9
    # -----------------------------
    pooled_rows = []

    for trial_idx, row in behav.iterrows():

        # Operand 1
        if 1 <= row["cue1"] <= 9:

            pooled_rows.append({
                "trial_idx": trial_idx,
                "trial": row["trial"],
                "operand_position": 1,
                "number": int(row["cue1"]),
                "onset": row["operand1_onset"]
            })

        # Operand 2
        if 1 <= row["cue2"] <= 9:

            pooled_rows.append({
                "trial_idx": trial_idx,
                "trial": row["trial"],
                "operand_position": 2,
                "number": int(row["cue2"]),
                "onset": row["operand2_onset"]
            })

    pooled = pd.DataFrame(
        pooled_rows
    )

    # -----------------------------
    # 0.05–0.95 s analysis window
    # Times are already in ms/sample units
    # -----------------------------
    pooled["onset_sample"] = (
        pooled["onset"]
        .round()
        .astype(int)
    )

    pooled["start_sample"] = (
        pooled["onset_sample"] + 50
    )

    pooled["end_sample"] = (
        pooled["onset_sample"] + 950
    )

    pooled["valid_window"] = (
        (pooled["start_sample"] >= 0)
        &
        (
            pooled["end_sample"]
            <= spikes.shape[1]
        )
    )

    return behav, pooled

In [78]:
def find_arithmetic_spike_file(subject):

    arithmetic_dir = (
        DATA_RAW
        / subject
        / "arithmetic"
    )

    candidates = [
        arithmetic_dir / "spikes.mat",
        arithmetic_dir / "spikesArithmetic.mat"
    ]

    for path in candidates:

        if path.exists():
            return path

    return None

In [82]:
subject_diagnostics = []

for subject in SUBJECTS:

    spike_path = find_arithmetic_spike_file(
        subject
    )

    if spike_path is None:

        subject_diagnostics.append({
            "subject": subject,
            "spike_file_found": False,
            "n_neurons": np.nan,
            "n_trials": np.nan,
            "n_presentations": np.nan,
            "min_class_count": np.nan,
            "valid_windows": np.nan
        })

        continue

    spikes_subj = (
        load_matlab_sparse_hdf5(
            spike_path,
            variable="spikes"
        )
    )

    behav_subj, pooled_subj = (
        prepare_arithmetic_subject(
            subject,
            spikes_subj
        )
    )

    class_counts = (
        pooled_subj["number"]
        .value_counts()
    )

    subject_diagnostics.append({
        "subject": subject,
        "spike_file_found": True,
        "n_neurons": spikes_subj.shape[0],
        "n_trials": len(behav_subj),
        "n_presentations": len(pooled_subj),
        "min_class_count": class_counts.min(),
        "valid_windows": pooled_subj[
            "valid_window"
        ].sum()
    })

diagnostics_df = pd.DataFrame(
    subject_diagnostics
)

display(diagnostics_df)

KeyError: "Variable 'spikes' not found in C:\\Users\\shafi\\number-simplex-reproduction\\data\\raw\\YFK\\arithmetic\\spikesArithmetic.mat. Available keys: ['#refs#', 'regionsVect', 'spikesArithmetic']"

In [83]:
from scipy import sparse

def load_arithmetic_spikes(filepath):

    filepath = Path(filepath)

    with h5py.File(filepath, "r") as f:

        possible_vars = [
            "spikes",
            "spikesArithmetic"
        ]

        variable = None

        for v in possible_vars:
            if v in f:
                variable = v
                break

        if variable is None:
            raise KeyError(
                f"No arithmetic spike variable found in {filepath}. "
                f"Available keys: {list(f.keys())}"
            )

        g = f[variable]

        required = {"ir", "jc", "data"}

        if not required.issubset(g.keys()):
            raise ValueError(
                f"{variable} is not in expected MATLAB sparse format."
            )

        ir = np.asarray(
            g["ir"]
        ).ravel().astype(np.int64)

        jc = np.asarray(
            g["jc"]
        ).ravel().astype(np.int64)

        data = np.asarray(
            g["data"]
        ).ravel()

        n_rows = int(
            np.asarray(
                g.attrs["MATLAB_sparse"]
            ).ravel()[0]
        )

        n_cols = len(jc) - 1

        mat = sparse.csc_matrix(
            (data, ir, jc),
            shape=(n_rows, n_cols)
        )

    return mat

In [84]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
from scipy import sparse


def find_arithmetic_spike_file(subject):
    arithmetic_dir = (
        DATA_RAW
        / subject
        / "arithmetic"
    )

    candidates = [
        arithmetic_dir / "spikes.mat",
        arithmetic_dir / "spikesArithmetic.mat"
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


def load_arithmetic_spikes(filepath):

    filepath = Path(filepath)

    with h5py.File(filepath, "r") as f:

        possible_vars = [
            "spikes",
            "spikesArithmetic"
        ]

        variable = None

        for v in possible_vars:
            if v in f:
                variable = v
                break

        if variable is None:
            raise KeyError(
                f"No arithmetic spike variable found in {filepath}. "
                f"Available keys: {list(f.keys())}"
            )

        g = f[variable]

        required = {"ir", "jc", "data"}

        if not required.issubset(g.keys()):
            raise ValueError(
                f"{variable} is not stored in the expected MATLAB sparse format."
            )

        ir = np.asarray(
            g["ir"]
        ).ravel().astype(np.int64)

        jc = np.asarray(
            g["jc"]
        ).ravel().astype(np.int64)

        data = np.asarray(
            g["data"]
        ).ravel()

        n_rows = int(
            np.asarray(
                g.attrs["MATLAB_sparse"]
            ).ravel()[0]
        )

        n_cols = len(jc) - 1

        spikes = sparse.csc_matrix(
            (data, ir, jc),
            shape=(n_rows, n_cols)
        )

    return spikes


def prepare_arithmetic_subject(subject, spikes):

    behavior_path = (
        DATA_RAW
        / subject
        / "arithmetic"
        / "photoBehavEvents.csv"
    )

    behav = pd.read_csv(
        behavior_path
    )

    behav = add_operand_onsets(
        behav,
        subject
    )

    pooled_rows = []

    for trial_idx, row in behav.iterrows():

        # Operand 1
        if 1 <= row["cue1"] <= 9:

            pooled_rows.append({
                "trial_idx": trial_idx,
                "trial": row["trial"],
                "operand_position": 1,
                "number": int(row["cue1"]),
                "onset": row["operand1_onset"]
            })

        # Operand 2
        if 1 <= row["cue2"] <= 9:

            pooled_rows.append({
                "trial_idx": trial_idx,
                "trial": row["trial"],
                "operand_position": 2,
                "number": int(row["cue2"]),
                "onset": row["operand2_onset"]
            })

    pooled = pd.DataFrame(
        pooled_rows
    )

    pooled["onset_sample"] = (
        pooled["onset"]
        .round()
        .astype(int)
    )

    pooled["start_sample"] = (
        pooled["onset_sample"] + 50
    )

    pooled["end_sample"] = (
        pooled["onset_sample"] + 950
    )

    pooled["valid_window"] = (
        (pooled["start_sample"] >= 0)
        &
        (
            pooled["end_sample"]
            <= spikes.shape[1]
        )
    )

    return behav, pooled


# ============================================================
# SUBJECT DIAGNOSTICS
# ============================================================

subject_diagnostics = []

for subject in SUBJECTS:

    print(f"Checking {subject}...")

    spike_path = find_arithmetic_spike_file(
        subject
    )

    if spike_path is None:

        subject_diagnostics.append({
            "subject": subject,
            "spike_file_found": False,
            "spike_filename": "",
            "n_neurons": np.nan,
            "n_trials": np.nan,
            "n_presentations": np.nan,
            "min_class_count": np.nan,
            "valid_windows": np.nan,
            "all_windows_valid": False
        })

        continue

    try:

        spikes_subj = load_arithmetic_spikes(
            spike_path
        )

        behav_subj, pooled_subj = (
            prepare_arithmetic_subject(
                subject,
                spikes_subj
            )
        )

        class_counts = (
            pooled_subj["number"]
            .value_counts()
        )

        n_valid_windows = int(
            pooled_subj[
                "valid_window"
            ].sum()
        )

        subject_diagnostics.append({
            "subject": subject,
            "spike_file_found": True,
            "spike_filename": spike_path.name,
            "n_neurons": spikes_subj.shape[0],
            "n_trials": len(behav_subj),
            "n_presentations": len(pooled_subj),
            "min_class_count": int(
                class_counts.min()
            ),
            "valid_windows": n_valid_windows,
            "all_windows_valid":
                n_valid_windows == len(pooled_subj)
        })

    except Exception as e:

        subject_diagnostics.append({
            "subject": subject,
            "spike_file_found": True,
            "spike_filename": spike_path.name,
            "n_neurons": np.nan,
            "n_trials": np.nan,
            "n_presentations": np.nan,
            "min_class_count": np.nan,
            "valid_windows": np.nan,
            "all_windows_valid": False,
            "error": str(e)
        })


diagnostics_df = pd.DataFrame(
    subject_diagnostics
)

display(
    diagnostics_df
)


print(
    "\nSubjects with spike files:",
    diagnostics_df[
        "spike_file_found"
    ].sum(),
    "/",
    len(SUBJECTS)
)

print(
    "Total loaded neurons:",
    diagnostics_df[
        "n_neurons"
    ].sum()
)

print(
    "Subjects with all windows valid:",
    diagnostics_df[
        "all_windows_valid"
    ].sum()
)

Checking YFF...
Checking YFI...
Checking YFJ...
Checking YFK...
Checking YFL...
Checking YFM...
Checking YFP...
Checking YFR...
Checking YFS...
Checking YFT...
Checking YFU...


,subject,spike_file_found,spike_filename,n_neurons,n_trials,n_presentations,min_class_count,valid_windows,all_windows_valid
0,YFF,True,spikes.mat,54,100,109,7,109,True
1,YFI,True,spikes.mat,38,100,104,9,104,True
2,YFJ,True,spikes.mat,62,100,114,4,114,True
3,YFK,True,spikesArithmetic.mat,68,100,114,4,114,True
4,YFL,True,spikesArithmetic.mat,94,100,114,4,114,True
5,YFM,True,spikesArithmetic.mat,94,100,114,4,114,True
6,YFP,True,spikes.mat,98,141,158,9,158,True
7,YFR,True,spikes.mat,78,160,266,20,266,True
8,YFS,True,spikes.mat,96,160,264,24,264,True
9,YFT,True,spikes.mat,89,300,477,43,477,True



Subjects with spike files: 11 / 11
Total loaded neurons: 849
Subjects with all windows valid: 11


In [85]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py


def decode_matlab_string(obj):
    """
    Convert a MATLAB HDF5 char/string dataset into a Python string.
    """

    arr = np.asarray(obj)

    # MATLAB char arrays are often uint16 character codes
    if np.issubdtype(arr.dtype, np.integer):
        chars = [
            chr(int(x))
            for x in arr.ravel()
            if int(x) != 0
        ]
        return "".join(chars).strip()

    # Byte strings
    if arr.dtype.kind in {"S", "a"}:
        return (
            b"".join(arr.ravel())
            .decode("utf-8")
            .strip()
        )

    # Unicode strings
    if arr.dtype.kind == "U":
        return "".join(
            arr.ravel()
        ).strip()

    return str(arr.squeeze()).strip()


def load_regions_from_mat(filepath):
    """
    Load regionsVect from MATLAB v7.3 HDF5 file.
    Handles MATLAB cell arrays stored as object references.
    """

    filepath = Path(filepath)

    with h5py.File(filepath, "r") as f:

        if "regionsVect" not in f:
            raise KeyError(
                f"'regionsVect' not found in {filepath}. "
                f"Available keys: {list(f.keys())}"
            )

        regions_obj = f["regionsVect"]

        regions = []

        # MATLAB cell array:
        # each element is an HDF5 object reference
        if regions_obj.dtype == h5py.ref_dtype:

            refs = np.asarray(
                regions_obj
            ).ravel()

            for ref in refs:

                if not ref:
                    regions.append("")
                    continue

                target = f[ref]

                regions.append(
                    decode_matlab_string(target)
                )

        else:

            # fallback in case some subject stores it differently
            arr = np.asarray(regions_obj)

            for item in arr.ravel():
                regions.append(
                    str(item).strip()
                )

    return regions


# ============================================================
# CHECK REGION COUNTS FOR EVERY SUBJECT
# ============================================================

region_rows = []

for subject in SUBJECTS:

    print(f"Reading regions for {subject}...")

    spike_path = find_arithmetic_spike_file(
        subject
    )

    spikes_subj = load_arithmetic_spikes(
        spike_path
    )

    regions = load_regions_from_mat(
        spike_path
    )

    print(
        f"  neurons = {spikes_subj.shape[0]}, "
        f"region labels = {len(regions)}"
    )

    if len(regions) != spikes_subj.shape[0]:
        print(
            "  WARNING: neuron count and region count differ!"
        )

    for neuron_idx, region in enumerate(regions):

        region_rows.append({
            "subject": subject,
            "neuron": neuron_idx,
            "region_raw": region
        })


regions_all = pd.DataFrame(
    region_rows
)

print("\nRaw region labels:")
print(
    regions_all["region_raw"]
    .value_counts(dropna=False)
)

print(
    "\nTotal region labels:",
    len(regions_all)
)

display(
    regions_all.head(20)
)

Reading regions for YFF...
  neurons = 54, region labels = 54
Reading regions for YFI...
  neurons = 38, region labels = 38
Reading regions for YFJ...
  neurons = 62, region labels = 62
Reading regions for YFK...
  neurons = 68, region labels = 68
Reading regions for YFL...
  neurons = 94, region labels = 94
Reading regions for YFM...
  neurons = 94, region labels = 94
Reading regions for YFP...
  neurons = 98, region labels = 98
Reading regions for YFR...
  neurons = 78, region labels = 78
Reading regions for YFS...
  neurons = 96, region labels = 96
Reading regions for YFT...
  neurons = 89, region labels = 89
Reading regions for YFU...
  neurons = 78, region labels = 78

Raw region labels:
region_raw
hpc          389
acc          142
ofc           82
amy           77
ent           70
thal          28
thalamus      26
para-hpc      18
OFC/vmPFC     17
Name: count, dtype: int64

Total region labels: 849


,subject,neuron,region_raw
0,YFF,0,acc
1,YFF,1,acc
2,YFF,2,acc
3,YFF,3,acc
4,YFF,4,acc
5,YFF,5,acc
6,YFF,6,acc
7,YFF,7,acc
8,YFF,8,ent
9,YFF,9,hpc


In [86]:
MTL_REGION_MAP = {
    "hpc": "HPC",
    "ent": "ENT",
    "amy": "AMY",
    "para-hpc": "PARA-HPC"
}

regions_all["is_mtl"] = (
    regions_all["region_raw"]
    .isin(MTL_REGION_MAP.keys())
)

regions_all["region"] = (
    regions_all["region_raw"]
    .map(MTL_REGION_MAP)
)

mtl_neurons = (
    regions_all.loc[
        regions_all["is_mtl"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("MTL neurons:", len(mtl_neurons))

print("\nMTL region counts:")
print(
    mtl_neurons["region_raw"]
    .value_counts()
)

print("\nNormalized region counts:")
print(
    mtl_neurons["region"]
    .value_counts()
)

print("\nMTL neurons per subject:")
print(
    mtl_neurons["subject"]
    .value_counts()
    .sort_index()
)

display(
    mtl_neurons.head(20)
)

MTL neurons: 554

MTL region counts:
region_raw
hpc         389
amy          77
ent          70
para-hpc     18
Name: count, dtype: int64

Normalized region counts:
region
HPC         389
AMY          77
ENT          70
PARA-HPC     18
Name: count, dtype: int64

MTL neurons per subject:
subject
YFF    37
YFI    29
YFJ    45
YFK    44
YFL    57
YFM    61
YFP    43
YFR    65
YFS    59
YFT    52
YFU    62
Name: count, dtype: int64


,subject,neuron,region_raw,is_mtl,region
0,YFF,8,ent,True,ENT
1,YFF,9,hpc,True,HPC
2,YFF,10,hpc,True,HPC
3,YFF,11,hpc,True,HPC
4,YFF,12,hpc,True,HPC
5,YFF,13,hpc,True,HPC
6,YFF,14,hpc,True,HPC
7,YFF,15,hpc,True,HPC
8,YFF,16,hpc,True,HPC
9,YFF,26,hpc,True,HPC


In [87]:
all_mtl_results = []

for subject in SUBJECTS:

    print("\n" + "=" * 60)
    print(f"SUBJECT: {subject}")
    print("=" * 60)

    # --------------------------------------------------
    # Load subject spike data
    # --------------------------------------------------
    spike_path = find_arithmetic_spike_file(
        subject
    )

    spikes_subj = load_arithmetic_spikes(
        spike_path
    )

    # --------------------------------------------------
    # Prepare behavior / pooled operand presentations
    # --------------------------------------------------
    behav_subj, pooled_subj = (
        prepare_arithmetic_subject(
            subject,
            spikes_subj
        )
    )

    # --------------------------------------------------
    # Get this subject's MTL neurons only
    # --------------------------------------------------
    subject_mtl = (
        mtl_neurons.loc[
            mtl_neurons["subject"] == subject
        ]
        .copy()
        .sort_values("neuron")
    )

    n_subject_mtl = len(
        subject_mtl
    )

    print(
        f"MTL neurons: {n_subject_mtl}"
    )

    # --------------------------------------------------
    # Analyze each MTL neuron
    # --------------------------------------------------
    for counter, (_, neuron_row) in enumerate(
        subject_mtl.iterrows(),
        start=1
    ):

        neuron_idx = int(
            neuron_row["neuron"]
        )

        region_raw = (
            neuron_row["region_raw"]
        )

        print(
            f"{subject}: "
            f"{counter}/{n_subject_mtl} "
            f"neuron={neuron_idx} "
            f"region={region_raw}"
        )

        result = analyze_one_temporal_neuron(
            spikes=spikes_subj,
            pooled_table=pooled_subj,
            neuron_idx=neuron_idx,
            bin_sizes=BIN_SIZES_MS,
            gammas=GAMMAS,
            n_permutations=N_PERMUTATIONS,
            random_state=0
        )

        result["subject"] = subject
        result["region_raw"] = region_raw
        result["region"] = (
            neuron_row["region"]
        )

        result["n_trials"] = len(
            behav_subj
        )

        result["n_presentations"] = len(
            pooled_subj
        )

        all_mtl_results.append(
            result
        )


# ======================================================
# BUILD FINAL TEMPORAL TABLE
# ======================================================

temporal_mtl = pd.DataFrame(
    all_mtl_results
)

# Put important columns first
column_order = [
    "subject",
    "neuron",
    "region_raw",
    "region",
    "n_trials",
    "n_presentations",
    "best_bin_ms",
    "best_gamma",
    "temporal_accuracy",
    "permutation_p",
    "n_perm_equal_or_better",
    "coding",
    "valid",
    "invalid_reason"
]

temporal_mtl = temporal_mtl[
    column_order
]

display(
    temporal_mtl.head(20)
)


SUBJECT: YFF
MTL neurons: 37
YFF: 1/37 neuron=8 region=ent
YFF: 2/37 neuron=9 region=hpc
YFF: 3/37 neuron=10 region=hpc
YFF: 4/37 neuron=11 region=hpc
YFF: 5/37 neuron=12 region=hpc
YFF: 6/37 neuron=13 region=hpc
YFF: 7/37 neuron=14 region=hpc
YFF: 8/37 neuron=15 region=hpc
YFF: 9/37 neuron=16 region=hpc
YFF: 10/37 neuron=26 region=hpc
YFF: 11/37 neuron=27 region=hpc
YFF: 12/37 neuron=28 region=hpc
YFF: 13/37 neuron=29 region=hpc
YFF: 14/37 neuron=30 region=hpc
YFF: 15/37 neuron=31 region=hpc
YFF: 16/37 neuron=32 region=hpc
YFF: 17/37 neuron=33 region=hpc
YFF: 18/37 neuron=34 region=hpc
YFF: 19/37 neuron=35 region=hpc
YFF: 20/37 neuron=36 region=hpc
YFF: 21/37 neuron=37 region=hpc
YFF: 22/37 neuron=38 region=hpc
YFF: 23/37 neuron=39 region=hpc
YFF: 24/37 neuron=40 region=hpc
YFF: 25/37 neuron=41 region=hpc
YFF: 26/37 neuron=42 region=hpc
YFF: 27/37 neuron=43 region=hpc
YFF: 28/37 neuron=44 region=hpc
YFF: 29/37 neuron=45 region=hpc
YFF: 30/37 neuron=46 region=hpc
YFF: 31/37 neuron=47 

,subject,neuron,region_raw,region,n_trials,n_presentations,best_bin_ms,best_gamma,temporal_accuracy,permutation_p,n_perm_equal_or_better,coding,valid,invalid_reason
0,YFF,8,ent,ENT,100,109,60.0,0.5,0.146789,0.288557,57.0,False,True,
1,YFF,9,hpc,HPC,100,109,90.0,0.2,0.183486,0.044776,8.0,True,True,
2,YFF,10,hpc,HPC,100,109,75.0,0.5,0.128440,0.318408,63.0,False,True,
3,YFF,11,hpc,HPC,100,109,450.0,0.2,0.155963,0.124378,24.0,False,True,
4,YFF,12,hpc,HPC,100,109,900.0,0.2,0.174312,0.034826,6.0,True,True,
5,YFF,13,hpc,HPC,100,109,75.0,0.5,0.183486,0.044776,8.0,True,True,
6,YFF,14,hpc,HPC,100,109,225.0,0.5,0.137615,0.228856,45.0,False,True,
7,YFF,15,hpc,HPC,100,109,60.0,0.2,0.128440,0.338308,67.0,False,True,
8,YFF,16,hpc,HPC,100,109,900.0,0.2,0.165138,0.074627,14.0,False,True,
9,YFF,26,hpc,HPC,100,109,90.0,0.8,0.137615,0.218905,43.0,False,True,


In [88]:
print(
    "Total MTL neurons:",
    len(temporal_mtl)
)

print(
    "Valid neurons:",
    temporal_mtl[
        "valid"
    ].sum()
)

print(
    "Invalid neurons:",
    (~temporal_mtl[
        "valid"
    ]).sum()
)

print(
    "Temporal coding neurons:",
    temporal_mtl[
        "coding"
    ].sum()
)

print(
    "Overall pooled coding percentage:",
    100
    * temporal_mtl["coding"].sum()
    / temporal_mtl["valid"].sum()
)

print("\nNaNs by column:")
print(
    temporal_mtl
    .isna()
    .sum()
)

Total MTL neurons: 554
Valid neurons: 544
Invalid neurons: 10
Temporal coding neurons: 160
Overall pooled coding percentage: 29.41176470588235

NaNs by column:
subject                    0
neuron                     0
region_raw                 0
region                     0
n_trials                   0
n_presentations            0
best_bin_ms                6
best_gamma                 6
temporal_accuracy          6
permutation_p             10
n_perm_equal_or_better    10
coding                     0
valid                      0
invalid_reason             0
dtype: int64


In [89]:
subject_temporal_summary = (
    temporal_mtl
    .groupby("subject")
    .agg(
        n_neurons=("neuron", "count"),
        n_valid=("valid", "sum"),
        n_coding=("coding", "sum")
    )
    .reset_index()
)

subject_temporal_summary[
    "coding_proportion"
] = (
    subject_temporal_summary[
        "n_coding"
    ]
    /
    subject_temporal_summary[
        "n_valid"
    ]
)

subject_temporal_summary[
    "coding_percentage"
] = (
    100
    * subject_temporal_summary[
        "coding_proportion"
    ]
)

display(
    subject_temporal_summary
)

print(
    "\nMean subject-level temporal coding percentage:",
    subject_temporal_summary[
        "coding_percentage"
    ].mean()
)

print(
    "SEM across subjects:",
    subject_temporal_summary[
        "coding_percentage"
    ].sem()
)

,subject,n_neurons,n_valid,n_coding,coding_proportion,coding_percentage
0,YFF,37,37,11,0.297297,29.729730
1,YFI,29,29,8,0.275862,27.586207
2,YFJ,45,45,13,0.288889,28.888889
3,YFK,44,42,11,0.261905,26.190476
4,YFL,57,56,13,0.232143,23.214286
5,YFM,61,58,17,0.293103,29.310345
6,YFP,43,41,10,0.243902,24.390244
7,YFR,65,64,17,0.265625,26.562500
8,YFS,59,58,21,0.362069,36.206897
9,YFT,52,52,16,0.307692,30.769231



Mean subject-level temporal coding percentage: 29.085961605860067
SEM across subjects: 1.3202794186317532


In [90]:
temporal_mtl.to_csv(
    TABLES / "arithmetic_temporal_mtl_neuron_results.csv",
    index=False
)

subject_temporal_summary.to_csv(
    TABLES / "arithmetic_temporal_subject_summary.csv",
    index=False
)

print("Saved:")
print(
    TABLES
    / "arithmetic_temporal_mtl_neuron_results.csv"
)

print(
    TABLES
    / "arithmetic_temporal_subject_summary.csv"
)

Saved:
C:\Users\shafi\number-simplex-reproduction\tables\arithmetic_temporal_mtl_neuron_results.csv
C:\Users\shafi\number-simplex-reproduction\tables\arithmetic_temporal_subject_summary.csv


In [91]:
# ============================================================
# FULL TEMPORAL RESULT DIAGNOSTICS
# ============================================================

# 1. Invalid neurons
invalid_neurons = (
    temporal_mtl.loc[
        ~temporal_mtl["valid"]
    ]
    .copy()
)

print("=" * 70)
print("INVALID NEURONS")
print("=" * 70)

display(
    invalid_neurons[
        [
            "subject",
            "neuron",
            "region_raw",
            "best_bin_ms",
            "best_gamma",
            "temporal_accuracy",
            "permutation_p",
            "invalid_reason"
        ]
    ]
)

print("\nInvalid neurons by subject:")
print(
    invalid_neurons[
        "subject"
    ]
    .value_counts()
    .sort_index()
)

print("\nInvalid reasons:")
print(
    invalid_neurons[
        "invalid_reason"
    ]
    .value_counts()
)


# ============================================================
# 2. SUBJECT-LEVEL CODING RESULTS
# ============================================================

subject_check = (
    temporal_mtl
    .groupby("subject")
    .agg(
        total_mtl_neurons=("neuron", "count"),
        valid_neurons=("valid", "sum"),
        coding_neurons=("coding", "sum")
    )
    .reset_index()
)

subject_check[
    "invalid_neurons"
] = (
    subject_check[
        "total_mtl_neurons"
    ]
    -
    subject_check[
        "valid_neurons"
    ]
)

subject_check[
    "coding_percent_valid_denominator"
] = (
    100
    * subject_check[
        "coding_neurons"
    ]
    /
    subject_check[
        "valid_neurons"
    ]
)

# Also calculate using ALL MTL neurons as denominator.
# This is useful because the paper reports 554 total neurons.
subject_check[
    "coding_percent_all_neurons"
] = (
    100
    * subject_check[
        "coding_neurons"
    ]
    /
    subject_check[
        "total_mtl_neurons"
    ]
)

print("\n" + "=" * 70)
print("SUBJECT-LEVEL RESULTS")
print("=" * 70)

display(
    subject_check
)


print(
    "\nMean using VALID-neuron denominator:",
    subject_check[
        "coding_percent_valid_denominator"
    ].mean()
)

print(
    "SEM using VALID-neuron denominator:",
    subject_check[
        "coding_percent_valid_denominator"
    ].sem()
)

print(
    "\nMean using ALL-neuron denominator:",
    subject_check[
        "coding_percent_all_neurons"
    ].mean()
)

print(
    "SEM using ALL-neuron denominator:",
    subject_check[
        "coding_percent_all_neurons"
    ].sem()
)


# ============================================================
# 3. PER-SUBJECT PARAMETER / P-VALUE INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CODING COUNTS BY SUBJECT")
print("=" * 70)

print(
    temporal_mtl.groupby(
        "subject"
    )["coding"].sum()
)


print("\nBest-bin counts:")
print(
    temporal_mtl.loc[
        temporal_mtl["valid"],
        "best_bin_ms"
    ]
    .value_counts()
    .sort_index()
)


print("\nBest-Gamma counts:")
print(
    temporal_mtl.loc[
        temporal_mtl["valid"],
        "best_gamma"
    ]
    .value_counts()
    .sort_index()
)

INVALID NEURONS


,subject,neuron,region_raw,best_bin_ms,best_gamma,temporal_accuracy,permutation_p,invalid_reason
114,YFK,27,hpc,60.0,0.2,0.166667,NaN,38 permutation accuracies were NaN
143,YFK,56,hpc,60.0,0.2,0.157895,NaN,1 permutation accuracies were NaN
163,YFL,27,hpc,NaN,NaN,NaN,NaN,all parameter combinations returned NaN
242,YFM,63,hpc,NaN,NaN,NaN,NaN,all parameter combinations returned NaN
254,YFM,75,hpc,NaN,NaN,NaN,NaN,all parameter combinations returned NaN
255,YFM,76,hpc,60.0,0.2,0.166667,NaN,13 permutation accuracies were NaN
285,YFP,41,hpc,NaN,NaN,NaN,NaN,all parameter combinations returned NaN
300,YFP,82,hpc,NaN,NaN,NaN,NaN,all parameter combinations returned NaN
330,YFR,14,hpc,NaN,NaN,NaN,NaN,all parameter combinations returned NaN
402,YFS,47,hpc,60.0,0.2,0.121212,NaN,25 permutation accuracies were NaN



Invalid neurons by subject:
subject
YFK    2
YFL    1
YFM    3
YFP    2
YFR    1
YFS    1
Name: count, dtype: int64

Invalid reasons:
invalid_reason
all parameter combinations returned NaN    6
38 permutation accuracies were NaN         1
1 permutation accuracies were NaN          1
13 permutation accuracies were NaN         1
25 permutation accuracies were NaN         1
Name: count, dtype: int64

SUBJECT-LEVEL RESULTS


,subject,total_mtl_neurons,valid_neurons,coding_neurons,invalid_neurons,coding_percent_valid_denominator,coding_percent_all_neurons
0,YFF,37,37,11,0,29.729730,29.729730
1,YFI,29,29,8,0,27.586207,27.586207
2,YFJ,45,45,13,0,28.888889,28.888889
3,YFK,44,42,11,2,26.190476,25.000000
4,YFL,57,56,13,1,23.214286,22.807018
5,YFM,61,58,17,3,29.310345,27.868852
6,YFP,43,41,10,2,24.390244,23.255814
7,YFR,65,64,17,1,26.562500,26.153846
8,YFS,59,58,21,1,36.206897,35.593220
9,YFT,52,52,16,0,30.769231,30.769231



Mean using VALID-neuron denominator: 29.085961605860067
SEM using VALID-neuron denominator: 1.3202794186317532

Mean using ALL-neuron denominator: 28.61359826610392
SEM using ALL-neuron denominator: 1.3810028260329825

CODING COUNTS BY SUBJECT
subject
YFF    11
YFI     8
YFJ    13
YFK    11
YFL    13
YFM    17
YFP    10
YFR    17
YFS    21
YFT    16
YFU    23
Name: coding, dtype: int64

Best-bin counts:
best_bin_ms
60.0     90
75.0     57
90.0     65
100.0    69
150.0    47
180.0    55
225.0    31
300.0    41
450.0    44
900.0    45
Name: count, dtype: int64

Best-Gamma counts:
best_gamma
0.2    327
0.5    105
0.8    112
Name: count, dtype: int64


In [92]:
paper_target = 24.80

subject_check_display = subject_check[
    [
        "subject",
        "total_mtl_neurons",
        "valid_neurons",
        "invalid_neurons",
        "coding_neurons",
        "coding_percent_all_neurons"
    ]
].copy()

display(subject_check_display)

print("\nCoding neurons total:")
print(subject_check_display["coding_neurons"].sum())

print("\nMTL neurons total:")
print(subject_check_display["total_mtl_neurons"].sum())

,subject,total_mtl_neurons,valid_neurons,invalid_neurons,coding_neurons,coding_percent_all_neurons
0,YFF,37,37,0,11,29.729730
1,YFI,29,29,0,8,27.586207
2,YFJ,45,45,0,13,28.888889
3,YFK,44,42,2,11,25.000000
4,YFL,57,56,1,13,22.807018
5,YFM,61,58,3,17,27.868852
6,YFP,43,41,2,10,23.255814
7,YFR,65,64,1,17,26.153846
8,YFS,59,58,1,21,35.593220
9,YFT,52,52,0,16,30.769231



Coding neurons total:
160

MTL neurons total:
554


In [93]:
# ============================================================
# CHECK BEHAVIORAL TRIAL EXCLUSION ACROSS ALL SUBJECTS
# ============================================================

exclusion_rows = []

for subject in SUBJECTS:

    behavior_path = (
        DATA_RAW
        / subject
        / "arithmetic"
        / "photoBehavEvents.csv"
    )

    behav = pd.read_csv(behavior_path)

    row = {
        "subject": subject,
        "n_trials": len(behav)
    }

    # --------------------------------------------------------
    # toExclude
    # --------------------------------------------------------
    if "toExclude" in behav.columns:

        row["toExclude_missing"] = (
            behav["toExclude"].isna().sum()
        )

        row["toExclude_0"] = (
            (behav["toExclude"] == 0).sum()
        )

        row["toExclude_1"] = (
            (behav["toExclude"] == 1).sum()
        )

        row["toExclude_other"] = (
            (
                behav["toExclude"].notna()
                & ~behav["toExclude"].isin([0, 1])
            ).sum()
        )

    # --------------------------------------------------------
    # correct
    # --------------------------------------------------------
    if "correct" in behav.columns:

        row["correct_0"] = (
            (behav["correct"] == 0).sum()
        )

        row["correct_1"] = (
            (behav["correct"] == 1).sum()
        )

        row["correct_missing"] = (
            behav["correct"].isna().sum()
        )

    exclusion_rows.append(row)


behavior_exclusion_check = pd.DataFrame(
    exclusion_rows
)

display(behavior_exclusion_check)


print("\n==============================")
print("TOTALS")
print("==============================")

numeric_cols = [
    c
    for c in behavior_exclusion_check.columns
    if c != "subject"
]

print(
    behavior_exclusion_check[
        numeric_cols
    ].sum()
)

,subject,n_trials,toExclude_missing,toExclude_0,toExclude_1,toExclude_other,correct_0,correct_1,correct_missing
0,YFF,100,0,100,0,0,16,84,0
1,YFI,100,0,99,1,0,13,87,0
2,YFJ,100,0,99,1,0,35,65,0
3,YFK,100,0,100,0,0,12,88,0
4,YFL,100,0,100,0,0,24,76,0
5,YFM,100,0,100,0,0,10,90,0
6,YFP,141,0,141,0,0,36,105,0
7,YFR,160,0,160,0,0,4,156,0
8,YFS,160,0,160,0,0,8,152,0
9,YFT,300,0,300,0,0,20,280,0



TOTALS
n_trials             1661
toExclude_missing       0
toExclude_0          1659
toExclude_1             2
toExclude_other         0
correct_0             278
correct_1            1383
correct_missing         0
dtype: int64


In [94]:
# ============================================================
# CHECK POOLED NUMBER PRESENTATIONS FOR EVERY SUBJECT
# ============================================================

class_count_rows = []

for subject in SUBJECTS:

    spike_path = find_arithmetic_spike_file(subject)
    spikes_subj = load_arithmetic_spikes(spike_path)

    behav_subj, pooled_subj = prepare_arithmetic_subject(
        subject,
        spikes_subj
    )

    counts = (
        pooled_subj["number"]
        .value_counts()
        .reindex(range(1, 10), fill_value=0)
        .sort_index()
    )

    row = {
        "subject": subject,
        "n_trials": len(behav_subj),
        "n_presentations": len(pooled_subj),
        "operand1_presentations":
            (pooled_subj["operand_position"] == 1).sum(),
        "operand2_presentations":
            (pooled_subj["operand_position"] == 2).sum(),
        "min_class_count": counts.min()
    }

    for number in range(1, 10):
        row[f"number_{number}"] = counts[number]

    class_count_rows.append(row)


presentation_check = pd.DataFrame(class_count_rows)

display(presentation_check)

print("\nTotal pooled presentations:")
print(presentation_check["n_presentations"].sum())

,subject,n_trials,n_presentations,operand1_presentations,operand2_presentations,min_class_count,number_1,number_2,number_3,number_4,number_5,number_6,number_7,number_8,number_9
0,YFF,100,109,55,54,7,10,15,11,7,7,16,17,12,14
1,YFI,100,104,55,49,9,10,11,11,11,9,10,10,23,9
2,YFJ,100,114,58,56,4,19,17,14,4,14,16,10,12,8
3,YFK,100,114,58,56,4,19,17,14,4,14,16,10,12,8
4,YFL,100,114,58,56,4,19,17,14,4,14,16,10,12,8
5,YFM,100,114,58,56,4,19,17,14,4,14,16,10,12,8
6,YFP,141,158,81,77,9,22,21,17,9,16,28,14,21,10
7,YFR,160,266,137,129,20,36,29,27,34,30,20,33,23,34
8,YFS,160,264,127,137,24,32,33,24,33,32,24,27,30,29
9,YFT,300,477,240,237,43,62,48,52,57,53,56,43,50,56



Total pooled presentations:
2328


In [95]:
# ============================================================
# CHECK WHETHER BEHAVIOR CSV FILES ARE DUPLICATES
# ============================================================

import hashlib

behavior_file_check = []

for subject in SUBJECTS:

    behavior_path = (
        DATA_RAW
        / subject
        / "arithmetic"
        / "photoBehavEvents.csv"
    )

    # Hash the exact file bytes
    with open(behavior_path, "rb") as f:
        file_bytes = f.read()

    md5_hash = hashlib.md5(
        file_bytes
    ).hexdigest()

    behav = pd.read_csv(
        behavior_path
    )

    behavior_file_check.append(
        {
            "subject": subject,
            "path": str(behavior_path),
            "n_trials": len(behav),
            "md5": md5_hash
        }
    )


behavior_file_check = pd.DataFrame(
    behavior_file_check
)

display(
    behavior_file_check
)


print("\n========================================")
print("DUPLICATE FILE HASHES")
print("========================================")

duplicate_hashes = (
    behavior_file_check
    .groupby("md5")["subject"]
    .apply(list)
)

for md5_hash, subjects in duplicate_hashes.items():

    if len(subjects) > 1:

        print(
            f"{subjects} -> {md5_hash}"
        )

,subject,path,n_trials,md5
0,YFF,C:\Users\shafi\number-simplex-reproduction\dat...,100,da21c9f5753e2484ab73a0b44807ca7d
1,YFI,C:\Users\shafi\number-simplex-reproduction\dat...,100,2fb1b853f8d754dc1fcdd23c6b15de82
2,YFJ,C:\Users\shafi\number-simplex-reproduction\dat...,100,1ea5b17d7e81707f5539491fe7725f51
3,YFK,C:\Users\shafi\number-simplex-reproduction\dat...,100,59dd7b31a8daa4721e33e88bb5ad8bbd
4,YFL,C:\Users\shafi\number-simplex-reproduction\dat...,100,98d649f9a90cdcf57098aac2b195001c
5,YFM,C:\Users\shafi\number-simplex-reproduction\dat...,100,274701bdfa04e37e80d85c6e9e553ed7
6,YFP,C:\Users\shafi\number-simplex-reproduction\dat...,141,01b2482620efe6c1e429560a1fb03c6b
7,YFR,C:\Users\shafi\number-simplex-reproduction\dat...,160,0938b6dfbe9a09cb0a3c926b4e1d266a
8,YFS,C:\Users\shafi\number-simplex-reproduction\dat...,160,614634a8e3f0d9d62c2fb346f1f3e784
9,YFT,C:\Users\shafi\number-simplex-reproduction\dat...,300,bc536b964c950eea6c970a588d3c09f6



DUPLICATE FILE HASHES


In [96]:
from sklearn.model_selection import StratifiedKFold

def matlab_gamma_lda_cv_inverse(
    X,
    y,
    gamma,
    random_state=0
):
    """
    Same implementation as our current matlab_gamma_lda_cv(),
    except Sigma_gamma is inverted with np.linalg.inv()
    instead of np.linalg.pinv().

    This lets us test the effect of that single change.
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.sort(np.unique(y))

    # Need all numbers 1-9
    if len(classes) != 9:
        return np.nan

    class_counts = pd.Series(y).value_counts()

    n_splits = min(
        10,
        int(class_counts.min())
    )

    if n_splits < 2:
        return np.nan

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    all_true = []
    all_pred = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # ----------------------------------------------------
        # Remove features with zero within-class variance
        # using TRAINING data only
        # ----------------------------------------------------

        within_class_variances = []

        for c in classes:

            Xc = X_train[y_train == c]

            v = np.var(
                Xc,
                axis=0,
                ddof=1
            )

            within_class_variances.append(v)

        within_class_variances = np.vstack(
            within_class_variances
        )

        keep = np.any(
            within_class_variances > 0,
            axis=0
        )

        if keep.sum() == 0:
            return np.nan

        X_train = X_train[:, keep]
        X_test = X_test[:, keep]

        # ----------------------------------------------------
        # Class means
        # ----------------------------------------------------

        means = {}

        for c in classes:

            means[c] = (
                X_train[y_train == c]
                .mean(axis=0)
            )

        # ----------------------------------------------------
        # Pooled within-class covariance
        # ----------------------------------------------------

        n_features = X_train.shape[1]

        scatter = np.zeros(
            (n_features, n_features),
            dtype=float
        )

        total_df = 0

        for c in classes:

            Xc = X_train[y_train == c]

            centered = (
                Xc - means[c]
            )

            scatter += (
                centered.T @ centered
            )

            total_df += (
                len(Xc) - 1
            )

        if total_df <= 0:
            return np.nan

        Sigma = (
            scatter / total_df
        )

        # ----------------------------------------------------
        # MATLAB-style Gamma regularization
        # ----------------------------------------------------

        Sigma_gamma = (
            (1 - gamma) * Sigma
            +
            gamma * np.diag(
                np.diag(Sigma)
            )
        )

        # ----------------------------------------------------
        # CONTROLLED CHANGE:
        # ordinary inverse instead of pseudoinverse
        # ----------------------------------------------------

        try:

            Sigma_inv = np.linalg.inv(
                Sigma_gamma
            )

        except np.linalg.LinAlgError:

            return np.nan

        # ----------------------------------------------------
        # Uniform prior
        # ----------------------------------------------------

        prior = (
            1.0 / len(classes)
        )

        # ----------------------------------------------------
        # LDA prediction
        # ----------------------------------------------------

        predictions = []

        for x in X_test:

            scores = []

            for c in classes:

                mu = means[c]

                score = (
                    x @ Sigma_inv @ mu
                    -
                    0.5
                    * mu
                    @ Sigma_inv
                    @ mu
                    +
                    np.log(prior)
                )

                scores.append(
                    score
                )

            predictions.append(
                classes[
                    np.argmax(scores)
                ]
            )

        all_true.extend(
            y_test
        )

        all_pred.extend(
            predictions
        )

    all_true = np.asarray(
        all_true
    )

    all_pred = np.asarray(
        all_pred
    )

    return np.mean(
        all_true == all_pred
    )

In [97]:
# Reload YFF
subject = "YFF"

spike_path = find_arithmetic_spike_file(
    subject
)

spikes_yff = load_arithmetic_spikes(
    spike_path
)

behav_yff, pooled_yff = (
    prepare_arithmetic_subject(
        subject,
        spikes_yff
    )
)

# Same neuron we previously tested
neuron_idx = 0
bin_ms = 90
gamma = 0.2

X_test, y_test = (
    make_pooled_temporal_features(
        spikes=spikes_yff,
        pooled_table=pooled_yff,
        neuron_idx=neuron_idx,
        bin_ms=bin_ms
    )
)

old_accuracy = matlab_gamma_lda_cv(
    X_test,
    y_test,
    gamma=gamma,
    random_state=0
)

new_accuracy = matlab_gamma_lda_cv_inverse(
    X_test,
    y_test,
    gamma=gamma,
    random_state=0
)

print("YFF neuron:", neuron_idx)
print("Bin:", bin_ms)
print("Gamma:", gamma)

print("\nPseudoinverse accuracy:")
print(old_accuracy)

print("\nOrdinary inverse accuracy:")
print(new_accuracy)

print("\nDifference:")
print(new_accuracy - old_accuracy)

YFF neuron: 0
Bin: 90
Gamma: 0.2

Pseudoinverse accuracy:
0.21100917431192662

Ordinary inverse accuracy:
0.21100917431192662

Difference:
0.0


In [98]:
# ============================================================
# CV PARTITION SENSITIVITY
# YFF neuron 0, fixed bin = 90 ms, Gamma = 0.2
# ============================================================

cv_seed_rows = []

for seed in range(50):

    acc = matlab_gamma_lda_cv(
        X_test,
        y_test,
        gamma=0.2,
        random_state=seed
    )

    cv_seed_rows.append(
        {
            "seed": seed,
            "accuracy": acc
        }
    )


cv_seed_check = pd.DataFrame(
    cv_seed_rows
)

display(cv_seed_check)

print("\nMinimum accuracy:")
print(cv_seed_check["accuracy"].min())

print("\nMaximum accuracy:")
print(cv_seed_check["accuracy"].max())

print("\nMean accuracy:")
print(cv_seed_check["accuracy"].mean())

print("\nSD across CV partitions:")
print(cv_seed_check["accuracy"].std())

print("\nNumber of unique accuracies:")
print(cv_seed_check["accuracy"].nunique())

print("\nAccuracy for seed 0:")
print(
    cv_seed_check.loc[
        cv_seed_check["seed"] == 0,
        "accuracy"
    ].iloc[0]
)

,seed,accuracy
0,0,0.211009
1,1,0.174312
2,2,0.174312
3,3,0.174312
4,4,0.247706
5,5,0.220183
6,6,0.165138
7,7,0.220183
8,8,0.183486
9,9,0.183486



Minimum accuracy:
0.13761467889908258

Maximum accuracy:
0.27522935779816515

Mean accuracy:
0.20844036697247706

SD across CV partitions:
0.028716550414153025

Number of unique accuracies:
14

Accuracy for seed 0:
0.21100917431192662


In [99]:
# ============================================================
# YFF: OBSERVED TEMPORAL DECODING ACROSS CV SEEDS
# No permutations yet
# ============================================================

subject = "YFF"

spike_path = find_arithmetic_spike_file(subject)
spikes_yff = load_arithmetic_spikes(spike_path)

behav_yff, pooled_yff = prepare_arithmetic_subject(
    subject,
    spikes_yff
)

yff_mtl = (
    mtl_neurons.loc[
        mtl_neurons["subject"] == subject
    ]
    .copy()
    .sort_values("neuron")
)

CV_SEEDS = [0, 1, 2, 3, 4]

seed_results = []

for seed in CV_SEEDS:

    print("\n" + "=" * 60)
    print(f"CV SEED: {seed}")
    print("=" * 60)

    for counter, (_, neuron_row) in enumerate(
        yff_mtl.iterrows(),
        start=1
    ):

        neuron_idx = int(neuron_row["neuron"])

        best_accuracy = np.nan
        best_bin = np.nan
        best_gamma = np.nan

        for bin_ms in BIN_SIZES_MS:

            X, y = make_pooled_temporal_features(
                spikes=spikes_yff,
                pooled_table=pooled_yff,
                neuron_idx=neuron_idx,
                bin_ms=bin_ms
            )

            for gamma in GAMMAS:

                try:
                    acc = matlab_gamma_lda_cv(
                        X,
                        y,
                        gamma=gamma,
                        random_state=seed
                    )
                except Exception:
                    acc = np.nan

                if (
                    not np.isnan(acc)
                    and (
                        np.isnan(best_accuracy)
                        or acc > best_accuracy
                    )
                ):
                    best_accuracy = acc
                    best_bin = bin_ms
                    best_gamma = gamma

        seed_results.append(
            {
                "seed": seed,
                "subject": subject,
                "neuron": neuron_idx,
                "region_raw": neuron_row["region_raw"],
                "best_accuracy": best_accuracy,
                "best_bin_ms": best_bin,
                "best_gamma": best_gamma
            }
        )

        print(
            f"{counter}/{len(yff_mtl)} "
            f"neuron={neuron_idx} "
            f"acc={best_accuracy:.4f} "
            f"bin={best_bin} "
            f"gamma={best_gamma}"
        )


yff_seed_results = pd.DataFrame(seed_results)

display(yff_seed_results.head(20))


CV SEED: 0
1/37 neuron=8 acc=0.1468 bin=60 gamma=0.5
2/37 neuron=9 acc=0.1835 bin=90 gamma=0.2
3/37 neuron=10 acc=0.1284 bin=75 gamma=0.5
4/37 neuron=11 acc=0.1560 bin=450 gamma=0.2
5/37 neuron=12 acc=0.1743 bin=900 gamma=0.2
6/37 neuron=13 acc=0.1835 bin=75 gamma=0.5
7/37 neuron=14 acc=0.1376 bin=225 gamma=0.5
8/37 neuron=15 acc=0.1284 bin=60 gamma=0.2
9/37 neuron=16 acc=0.1651 bin=900 gamma=0.2
10/37 neuron=26 acc=0.1376 bin=90 gamma=0.8
11/37 neuron=27 acc=0.1101 bin=75 gamma=0.2
12/37 neuron=28 acc=0.1927 bin=60 gamma=0.2
13/37 neuron=29 acc=0.1009 bin=75 gamma=0.2
14/37 neuron=30 acc=0.1927 bin=100 gamma=0.8
15/37 neuron=31 acc=0.0917 bin=150 gamma=0.2
16/37 neuron=32 acc=0.2202 bin=300 gamma=0.2
17/37 neuron=33 acc=0.2018 bin=60 gamma=0.2
18/37 neuron=34 acc=0.2202 bin=180 gamma=0.5
19/37 neuron=35 acc=0.1651 bin=150 gamma=0.2
20/37 neuron=36 acc=0.0642 bin=150 gamma=0.2
21/37 neuron=37 acc=0.1468 bin=60 gamma=0.2
22/37 neuron=38 acc=0.1651 bin=75 gamma=0.2
23/37 neuron=39 acc=0

,seed,subject,neuron,region_raw,best_accuracy,best_bin_ms,best_gamma
0,0,YFF,8,ent,0.146789,60,0.5
1,0,YFF,9,hpc,0.183486,90,0.2
2,0,YFF,10,hpc,0.128440,75,0.5
3,0,YFF,11,hpc,0.155963,450,0.2
4,0,YFF,12,hpc,0.174312,900,0.2
5,0,YFF,13,hpc,0.183486,75,0.5
6,0,YFF,14,hpc,0.137615,225,0.5
7,0,YFF,15,hpc,0.128440,60,0.2
8,0,YFF,16,hpc,0.165138,900,0.2
9,0,YFF,26,hpc,0.137615,90,0.8


In [100]:
# ============================================================
# SUMMARIZE CV-SEED SENSITIVITY
# ============================================================

accuracy_stability = (
    yff_seed_results
    .groupby("neuron")
    .agg(
        min_accuracy=("best_accuracy", "min"),
        max_accuracy=("best_accuracy", "max"),
        mean_accuracy=("best_accuracy", "mean"),
        sd_accuracy=("best_accuracy", "std"),
        n_unique_bins=("best_bin_ms", "nunique"),
        n_unique_gammas=("best_gamma", "nunique")
    )
    .reset_index()
)

accuracy_stability["accuracy_range"] = (
    accuracy_stability["max_accuracy"]
    -
    accuracy_stability["min_accuracy"]
)

display(
    accuracy_stability.sort_values(
        "accuracy_range",
        ascending=False
    )
)

print(
    "\nMean accuracy range across YFF neurons:",
    accuracy_stability["accuracy_range"].mean()
)

print(
    "Median accuracy range:",
    accuracy_stability["accuracy_range"].median()
)

print(
    "\nNeurons whose selected bin changed across seeds:",
    (accuracy_stability["n_unique_bins"] > 1).sum(),
    "/",
    len(accuracy_stability)
)

print(
    "Neurons whose selected Gamma changed across seeds:",
    (accuracy_stability["n_unique_gammas"] > 1).sum(),
    "/",
    len(accuracy_stability)
)

,neuron,min_accuracy,max_accuracy,mean_accuracy,sd_accuracy,n_unique_bins,n_unique_gammas,accuracy_range
2,10,0.128440,0.211009,0.174312,0.030428,2,3,0.082569
25,42,0.128440,0.192661,0.159633,0.023924,4,3,0.064220
0,8,0.128440,0.183486,0.154128,0.019889,2,3,0.055046
8,16,0.146789,0.201835,0.172477,0.023747,3,2,0.055046
21,38,0.165138,0.220183,0.194495,0.024617,3,3,0.055046
14,31,0.091743,0.146789,0.115596,0.020100,1,1,0.055046
35,52,0.137615,0.183486,0.165138,0.017164,3,2,0.045872
4,12,0.128440,0.174312,0.143119,0.019024,5,2,0.045872
20,37,0.119266,0.165138,0.148624,0.017647,2,3,0.045872
15,32,0.174312,0.220183,0.194495,0.017647,4,2,0.045872



Mean accuracy range across YFF neurons: 0.033969749566079836
Median accuracy range: 0.03669724770642202

Neurons whose selected bin changed across seeds: 31 / 37
Neurons whose selected Gamma changed across seeds: 31 / 37


In [101]:
# ============================================================
# YFF FULL CODING ANALYSIS ACROSS 5 CV SEEDS
# ============================================================

YFF_CV_SEEDS = [0, 1, 2, 3, 4]

yff_seed_coding_results = []

for seed in YFF_CV_SEEDS:

    print("\n" + "=" * 70)
    print(f"YFF — FULL PERMUTATION ANALYSIS — SEED {seed}")
    print("=" * 70)

    for counter, (_, neuron_row) in enumerate(
        yff_mtl.iterrows(),
        start=1
    ):

        neuron_idx = int(
            neuron_row["neuron"]
        )

        result = analyze_one_temporal_neuron(
            spikes=spikes_yff,
            pooled_table=pooled_yff,
            neuron_idx=neuron_idx,
            bin_sizes=BIN_SIZES_MS,
            gammas=GAMMAS,
            n_permutations=200,
            random_state=seed
        )

        result["seed"] = seed
        result["subject"] = "YFF"
        result["region_raw"] = neuron_row["region_raw"]

        yff_seed_coding_results.append(
            result
        )

        print(
            f"{counter}/{len(yff_mtl)} "
            f"neuron={neuron_idx} "
            f"coding={result['coding']} "
            f"p={result['permutation_p']}"
        )


yff_seed_coding = pd.DataFrame(
    yff_seed_coding_results
)

display(
    yff_seed_coding.head()
)


YFF — FULL PERMUTATION ANALYSIS — SEED 0
1/37 neuron=8 coding=False p=0.2885572139303483
2/37 neuron=9 coding=True p=0.04477611940298507
3/37 neuron=10 coding=False p=0.31840796019900497
4/37 neuron=11 coding=False p=0.12437810945273632
5/37 neuron=12 coding=True p=0.03482587064676617
6/37 neuron=13 coding=True p=0.04477611940298507
7/37 neuron=14 coding=False p=0.22885572139303484
8/37 neuron=15 coding=False p=0.3383084577114428
9/37 neuron=16 coding=False p=0.07462686567164178
10/37 neuron=26 coding=False p=0.21890547263681592
11/37 neuron=27 coding=False p=0.35323383084577115
12/37 neuron=28 coding=True p=0.01990049751243781
13/37 neuron=29 coding=False p=0.5074626865671642
14/37 neuron=30 coding=True p=0.004975124378109453
15/37 neuron=31 coding=False p=0.6417910447761194
16/37 neuron=32 coding=True p=0.004975124378109453
17/37 neuron=33 coding=True p=0.004975124378109453
18/37 neuron=34 coding=True p=0.024875621890547265
19/37 neuron=35 coding=False p=0.0845771144278607
20/37 neu

,neuron,best_bin_ms,best_gamma,temporal_accuracy,permutation_p,n_perm_equal_or_better,coding,valid,invalid_reason,seed,subject,region_raw
0,8,60,0.5,0.146789,0.288557,57,False,True,,0,YFF,ent
1,9,90,0.2,0.183486,0.044776,8,True,True,,0,YFF,hpc
2,10,75,0.5,0.128440,0.318408,63,False,True,,0,YFF,hpc
3,11,450,0.2,0.155963,0.124378,24,False,True,,0,YFF,hpc
4,12,900,0.2,0.174312,0.034826,6,True,True,,0,YFF,hpc


In [102]:
# ============================================================
# YFF CODING PROPORTION FOR EACH CV SEED
# ============================================================

yff_seed_summary = (
    yff_seed_coding
    .groupby("seed")
    .agg(
        total_neurons=("neuron", "count"),
        valid_neurons=("valid", "sum"),
        coding_neurons=("coding", "sum")
    )
    .reset_index()
)

yff_seed_summary["coding_percentage_all"] = (
    100
    * yff_seed_summary["coding_neurons"]
    / yff_seed_summary["total_neurons"]
)

yff_seed_summary["coding_percentage_valid"] = (
    100
    * yff_seed_summary["coding_neurons"]
    / yff_seed_summary["valid_neurons"]
)

display(yff_seed_summary)

print("\nCoding percentage range across seeds:")
print(
    yff_seed_summary["coding_percentage_all"].min(),
    "to",
    yff_seed_summary["coding_percentage_all"].max()
)

print("\nCoding-neuron count range:")
print(
    yff_seed_summary["coding_neurons"].min(),
    "to",
    yff_seed_summary["coding_neurons"].max()
)

,seed,total_neurons,valid_neurons,coding_neurons,coding_percentage_all,coding_percentage_valid
0,0,37,37,11,29.729730,29.729730
1,1,37,37,9,24.324324,24.324324
2,2,37,37,10,27.027027,27.027027
3,3,37,37,12,32.432432,32.432432
4,4,37,37,11,29.729730,29.729730



Coding percentage range across seeds:
24.324324324324323 to 32.432432432432435

Coding-neuron count range:
9 to 12


Data/regions/windows look correct
	​

$$ \boxed{\text{Our seed-0 result: }29.09\%\pm1.32\%} $$
Paper: 24.80%±1.73%

In [103]:
from sklearn.model_selection import StratifiedKFold


def make_stratified_folds(y, random_state=0):
    """
    Create ONE stratified CV partition from the original labels.
    These train/test indices can then be reused.
    """

    y = np.asarray(y)

    classes = np.sort(np.unique(y))

    if len(classes) != 9:
        return None

    class_counts = pd.Series(y).value_counts()

    n_splits = min(
        10,
        int(class_counts.min())
    )

    if n_splits < 2:
        return None

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    # X is irrelevant for generating stratified indices
    dummy_X = np.zeros((len(y), 1))

    folds = list(
        cv.split(dummy_X, y)
    )

    return folds


def matlab_gamma_lda_fixed_folds(
    X,
    y,
    gamma,
    folds
):
    """
    MATLAB-style regularized LDA evaluated on a supplied,
    FIXED set of train/test folds.
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.sort(np.unique(y))

    if len(classes) != 9:
        return np.nan

    all_true = []
    all_pred = []

    for train_idx, test_idx in folds:

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # Every training fold must still contain all classes
        if len(np.unique(y_train)) != 9:
            return np.nan

        # ----------------------------------------------------
        # Remove zero within-class-variance features
        # using training data only
        # ----------------------------------------------------

        within_class_variances = []

        for c in classes:

            Xc = X_train[y_train == c]

            if len(Xc) < 2:
                return np.nan

            v = np.var(
                Xc,
                axis=0,
                ddof=1
            )

            within_class_variances.append(v)

        within_class_variances = np.vstack(
            within_class_variances
        )

        keep = np.any(
            within_class_variances > 0,
            axis=0
        )

        if keep.sum() == 0:
            return np.nan

        X_train = X_train[:, keep]
        X_test = X_test[:, keep]

        # ----------------------------------------------------
        # Class means
        # ----------------------------------------------------

        means = {}

        for c in classes:

            means[c] = (
                X_train[y_train == c]
                .mean(axis=0)
            )

        # ----------------------------------------------------
        # Pooled within-class covariance
        # ----------------------------------------------------

        n_features = X_train.shape[1]

        scatter = np.zeros(
            (n_features, n_features),
            dtype=float
        )

        total_df = 0

        for c in classes:

            Xc = X_train[y_train == c]

            centered = (
                Xc - means[c]
            )

            scatter += (
                centered.T @ centered
            )

            total_df += (
                len(Xc) - 1
            )

        if total_df <= 0:
            return np.nan

        Sigma = (
            scatter / total_df
        )

        # ----------------------------------------------------
        # Gamma regularization
        # ----------------------------------------------------

        Sigma_gamma = (
            (1 - gamma) * Sigma
            +
            gamma * np.diag(
                np.diag(Sigma)
            )
        )

        Sigma_inv = np.linalg.pinv(
            Sigma_gamma
        )

        prior = 1.0 / len(classes)

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        predictions = []

        for x in X_test:

            scores = []

            for c in classes:

                mu = means[c]

                score = (
                    x @ Sigma_inv @ mu
                    -
                    0.5
                    * mu
                    @ Sigma_inv
                    @ mu
                    +
                    np.log(prior)
                )

                scores.append(score)

            predictions.append(
                classes[np.argmax(scores)]
            )

        all_true.extend(y_test)
        all_pred.extend(predictions)

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)

    return np.mean(
        all_true == all_pred
    )

In [104]:
# ============================================================
# SANITY CHECK — YFF NEURON 0
# ============================================================

neuron_idx = 0
bin_ms = 90
gamma = 0.2

X0, y0 = make_pooled_temporal_features(
    spikes=spikes_yff,
    pooled_table=pooled_yff,
    neuron_idx=neuron_idx,
    bin_ms=bin_ms
)

folds0 = make_stratified_folds(
    y0,
    random_state=0
)

old_acc = matlab_gamma_lda_cv(
    X0,
    y0,
    gamma=gamma,
    random_state=0
)

fixed_fold_acc = matlab_gamma_lda_fixed_folds(
    X0,
    y0,
    gamma=gamma,
    folds=folds0
)

print("Old decoder accuracy:")
print(old_acc)

print("\nFixed-fold decoder accuracy:")
print(fixed_fold_acc)

print("\nDifference:")
print(fixed_fold_acc - old_acc)

print("\nNumber of folds:")
print(len(folds0))

Old decoder accuracy:
0.21100917431192662

Fixed-fold decoder accuracy:
0.21100917431192662

Difference:
0.0

Number of folds:
7


In [105]:
# ============================================================
# PERMUTATION TEST WITH FIXED CV FOLDS
# YFF neuron 0
# ============================================================

def permutation_test_fixed_folds(
    X,
    y,
    gamma,
    folds,
    n_permutations=200,
    random_state=0
):
    """
    Observed and shuffled-label decoding use the SAME
    train/test indices.
    """

    observed_accuracy = matlab_gamma_lda_fixed_folds(
        X=X,
        y=y,
        gamma=gamma,
        folds=folds
    )

    if np.isnan(observed_accuracy):
        return {
            "observed_accuracy": np.nan,
            "permutation_p": np.nan,
            "n_equal_or_better": np.nan,
            "n_nan_permutations": np.nan,
            "valid": False
        }

    rng = np.random.default_rng(
        random_state
    )

    permuted_accuracies = []

    for _ in range(n_permutations):

        y_shuffled = rng.permutation(
            y
        )

        perm_acc = matlab_gamma_lda_fixed_folds(
            X=X,
            y=y_shuffled,
            gamma=gamma,
            folds=folds
        )

        permuted_accuracies.append(
            perm_acc
        )

    permuted_accuracies = np.asarray(
        permuted_accuracies,
        dtype=float
    )

    n_nan = np.isnan(
        permuted_accuracies
    ).sum()

    if n_nan > 0:
        return {
            "observed_accuracy": observed_accuracy,
            "permutation_p": np.nan,
            "n_equal_or_better": np.nan,
            "n_nan_permutations": int(n_nan),
            "valid": False
        }

    n_equal_or_better = np.sum(
        permuted_accuracies
        >= observed_accuracy
    )

    p_value = (
        1 + n_equal_or_better
    ) / (
        1 + n_permutations
    )

    return {
        "observed_accuracy": float(
            observed_accuracy
        ),
        "permutation_p": float(
            p_value
        ),
        "n_equal_or_better": int(
            n_equal_or_better
        ),
        "n_nan_permutations": 0,
        "valid": True
    }


fixed_perm_result = permutation_test_fixed_folds(
    X=X0,
    y=y0,
    gamma=0.2,
    folds=folds0,
    n_permutations=200,
    random_state=0
)

print("Observed accuracy:")
print(
    fixed_perm_result[
        "observed_accuracy"
    ]
)

print("\nFixed-fold permutation p-value:")
print(
    fixed_perm_result[
        "permutation_p"
    ]
)

print("\nPermutations >= observed:")
print(
    fixed_perm_result[
        "n_equal_or_better"
    ]
)

print("\nNaN permutations:")
print(
    fixed_perm_result[
        "n_nan_permutations"
    ]
)

print("\nCoding:")
print(
    fixed_perm_result[
        "permutation_p"
    ] < 0.05
)

Observed accuracy:
0.21100917431192662

Fixed-fold permutation p-value:
nan

Permutations >= observed:
nan

NaN permutations:
1

Coding:
False


In [106]:
# ============================================================
# DIAGNOSE NEURONS WHERE ALL 30 TEMPORAL MODELS RETURNED NaN
# ============================================================

from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------
# Identify only neurons with:
# "all parameter combinations returned NaN"
# ------------------------------------------------------------

all_nan_neurons = (
    temporal_mtl.loc[
        temporal_mtl["invalid_reason"]
        == "all parameter combinations returned NaN"
    ]
    [
        ["subject", "neuron", "region_raw", "region"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Number of all-NaN neurons:")
print(len(all_nan_neurons))

display(all_nan_neurons)


# ============================================================
# Helper: inspect what happens inside CV
# ============================================================

def diagnose_temporal_features(
    X,
    y,
    random_state=0
):
    """
    Inspect feature variability and the number of predictors
    surviving our within-class-variance rule in each CV fold.
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.sort(np.unique(y))

    result = {
        "n_samples": len(y),
        "n_classes": len(classes),
        "n_features": X.shape[1],
        "total_spike_count": X.sum(),
        "nonzero_presentations": np.sum(
            X.sum(axis=1) > 0
        ),
        "globally_nonzero_features": np.sum(
            np.var(X, axis=0) > 0
        ),
        "min_kept_features": np.nan,
        "max_kept_features": np.nan,
        "zero_feature_folds": np.nan,
        "n_folds": np.nan
    }

    if len(classes) != 9:
        return result

    class_counts = pd.Series(y).value_counts()

    n_splits = min(
        10,
        int(class_counts.min())
    )

    result["n_folds"] = n_splits

    if n_splits < 2:
        return result

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    kept_counts = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        y_train = y[train_idx]

        within_class_variances = []

        for c in classes:

            Xc = X_train[y_train == c]

            v = np.var(
                Xc,
                axis=0,
                ddof=1
            )

            within_class_variances.append(v)

        within_class_variances = np.vstack(
            within_class_variances
        )

        keep = np.any(
            within_class_variances > 0,
            axis=0
        )

        kept_counts.append(
            int(keep.sum())
        )

    result["min_kept_features"] = min(
        kept_counts
    )

    result["max_kept_features"] = max(
        kept_counts
    )

    result["zero_feature_folds"] = sum(
        k == 0
        for k in kept_counts
    )

    return result


# ============================================================
# Run diagnostics for each failed neuron and every bin size
# ============================================================

diagnostic_rows = []

for _, neuron_info in all_nan_neurons.iterrows():

    subject = neuron_info["subject"]
    neuron_idx = int(
        neuron_info["neuron"]
    )

    print(
        f"\nDiagnosing {subject} "
        f"neuron {neuron_idx} "
        f"({neuron_info['region_raw']})"
    )

    spike_path = find_arithmetic_spike_file(
        subject
    )

    spikes_subj = load_arithmetic_spikes(
        spike_path
    )

    behav_subj, pooled_subj = (
        prepare_arithmetic_subject(
            subject,
            spikes_subj
        )
    )

    for bin_ms in BIN_SIZES_MS:

        X, y = make_pooled_temporal_features(
            spikes=spikes_subj,
            pooled_table=pooled_subj,
            neuron_idx=neuron_idx,
            bin_ms=bin_ms
        )

        d = diagnose_temporal_features(
            X=X,
            y=y,
            random_state=0
        )

        # Try all three Gamma values
        gamma_results = {}

        for gamma in GAMMAS:

            try:
                acc = matlab_gamma_lda_cv(
                    X,
                    y,
                    gamma=gamma,
                    random_state=0
                )
            except Exception:
                acc = np.nan

            gamma_results[
                f"gamma_{gamma}_accuracy"
            ] = acc

        diagnostic_rows.append(
            {
                "subject": subject,
                "neuron": neuron_idx,
                "region_raw":
                    neuron_info["region_raw"],
                "bin_ms": bin_ms,
                **d,
                **gamma_results
            }
        )


failed_neuron_diagnostics = pd.DataFrame(
    diagnostic_rows
)

display(
    failed_neuron_diagnostics
)

Number of all-NaN neurons:
6


,subject,neuron,region_raw,region
0,YFL,27,hpc,HPC
1,YFM,63,hpc,HPC
2,YFM,75,hpc,HPC
3,YFP,41,hpc,HPC
4,YFP,82,hpc,HPC
5,YFR,14,hpc,HPC



Diagnosing YFL neuron 27 (hpc)

Diagnosing YFM neuron 63 (hpc)

Diagnosing YFM neuron 75 (hpc)

Diagnosing YFP neuron 41 (hpc)

Diagnosing YFP neuron 82 (hpc)

Diagnosing YFR neuron 14 (hpc)


,subject,neuron,region_raw,bin_ms,n_samples,n_classes,n_features,total_spike_count,nonzero_presentations,globally_nonzero_features,min_kept_features,max_kept_features,zero_feature_folds,n_folds,gamma_0.2_accuracy,gamma_0.5_accuracy,gamma_0.8_accuracy
0,YFL,27,hpc,60,114,9,15,0.0,0,0,0,0,4,4,NaN,NaN,NaN
1,YFL,27,hpc,75,114,9,12,0.0,0,0,0,0,4,4,NaN,NaN,NaN
2,YFL,27,hpc,90,114,9,10,0.0,0,0,0,0,4,4,NaN,NaN,NaN
3,YFL,27,hpc,100,114,9,9,0.0,0,0,0,0,4,4,NaN,NaN,NaN
4,YFL,27,hpc,150,114,9,6,0.0,0,0,0,0,4,4,NaN,NaN,NaN
5,YFL,27,hpc,180,114,9,5,0.0,0,0,0,0,4,4,NaN,NaN,NaN
6,YFL,27,hpc,225,114,9,4,0.0,0,0,0,0,4,4,NaN,NaN,NaN
7,YFL,27,hpc,300,114,9,3,0.0,0,0,0,0,4,4,NaN,NaN,NaN
8,YFL,27,hpc,450,114,9,2,0.0,0,0,0,0,4,4,NaN,NaN,NaN
9,YFL,27,hpc,900,114,9,1,0.0,0,0,0,0,4,4,NaN,NaN,NaN


In [107]:
# ============================================================
# ONE-ROW SUMMARY PER FAILED NEURON
# ============================================================

failed_neuron_summary = (
    failed_neuron_diagnostics
    .groupby(
        ["subject", "neuron", "region_raw"]
    )
    .agg(
        n_samples=("n_samples", "first"),

        # Total spikes are identical across bin sizes
        # because all bins cover the same 900-ms window.
        total_spikes=("total_spike_count", "first"),

        nonzero_presentations=(
            "nonzero_presentations",
            "first"
        ),

        smallest_min_kept_features=(
            "min_kept_features",
            "min"
        ),

        largest_max_kept_features=(
            "max_kept_features",
            "max"
        ),

        max_zero_feature_folds=(
            "zero_feature_folds",
            "max"
        )
    )
    .reset_index()
)

failed_neuron_summary[
    "percent_presentations_with_spikes"
] = (
    100
    * failed_neuron_summary[
        "nonzero_presentations"
    ]
    / failed_neuron_summary[
        "n_samples"
    ]
)

display(
    failed_neuron_summary
)

,subject,neuron,region_raw,n_samples,total_spikes,nonzero_presentations,smallest_min_kept_features,largest_max_kept_features,max_zero_feature_folds,percent_presentations_with_spikes
0,YFL,27,hpc,114,0.0,0,0,0,4,0.000000
1,YFM,63,hpc,114,1.0,1,0,1,1,0.877193
2,YFM,75,hpc,114,4.0,3,0,3,1,2.631579
3,YFP,41,hpc,158,1.0,1,0,1,1,0.632911
4,YFP,82,hpc,158,1.0,1,0,1,1,0.632911
5,YFR,14,hpc,266,1.0,1,0,1,1,0.375940


## lets proceed FR analysiss

In [108]:
def analyze_one_firing_rate_neuron(
    spikes,
    pooled_table,
    neuron_idx,
    n_permutations=200,
    random_state=0
):
    """
    Arithmetic firing-rate decoding.

    Each operand presentation is represented by ONE feature:
    total spike count from 0.05 to 0.95 s after operand onset.

    This is equivalent to the 900-ms one-bin representation.
    """

    # ========================================================
    # BUILD ONE-DIMENSIONAL FIRING-RATE FEATURE
    # ========================================================

    X, y = make_pooled_temporal_features(
        spikes=spikes,
        pooled_table=pooled_table,
        neuron_idx=neuron_idx,
        bin_ms=900
    )

    # Gamma is irrelevant in one dimension.
    # Use 0.2 simply to pass through the existing decoder.
    gamma = 0.2

    # ========================================================
    # OBSERVED CROSS-VALIDATED ACCURACY
    # ========================================================

    try:

        observed_accuracy = matlab_gamma_lda_cv(
            X,
            y,
            gamma=gamma,
            random_state=random_state
        )

    except Exception:

        observed_accuracy = np.nan

    if np.isnan(observed_accuracy):

        return {
            "neuron": neuron_idx,
            "fr_accuracy": np.nan,
            "permutation_p": np.nan,
            "n_perm_equal_or_better": np.nan,
            "coding": False,
            "valid": False,
            "invalid_reason":
                "observed firing-rate decoding returned NaN"
        }

    # ========================================================
    # PERMUTATION TEST
    # ========================================================

    rng = np.random.default_rng(
        random_state
    )

    permuted_accuracies = []

    for permutation_idx in range(
        n_permutations
    ):

        y_shuffled = rng.permutation(
            y
        )

        try:

            perm_acc = matlab_gamma_lda_cv(
                X,
                y_shuffled,
                gamma=gamma,
                random_state=random_state
            )

        except Exception:

            perm_acc = np.nan

        permuted_accuracies.append(
            perm_acc
        )

    permuted_accuracies = np.asarray(
        permuted_accuracies,
        dtype=float
    )

    n_nan_perm = np.isnan(
        permuted_accuracies
    ).sum()

    # ========================================================
    # INVALID PERMUTATION CASE
    # ========================================================

    if n_nan_perm > 0:

        return {
            "neuron": neuron_idx,
            "fr_accuracy":
                float(observed_accuracy),
            "permutation_p": np.nan,
            "n_perm_equal_or_better": np.nan,
            "coding": False,
            "valid": False,
            "invalid_reason":
                f"{n_nan_perm} permutation accuracies were NaN"
        }

    # ========================================================
    # PAPER'S PERMUTATION P-VALUE
    # ========================================================

    n_equal_or_better = np.sum(
        permuted_accuracies
        >= observed_accuracy
    )

    p_value = (
        1 + n_equal_or_better
    ) / (
        1 + n_permutations
    )

    return {
        "neuron": neuron_idx,
        "fr_accuracy":
            float(observed_accuracy),
        "permutation_p":
            float(p_value),
        "n_perm_equal_or_better":
            int(n_equal_or_better),
        "coding":
            bool(p_value < 0.05),
        "valid": True,
        "invalid_reason": ""
    }

In [109]:
fr_neuron0 = analyze_one_firing_rate_neuron(
    spikes=spikes_yff,
    pooled_table=pooled_yff,
    neuron_idx=0,
    n_permutations=200,
    random_state=0
)

print("YFF neuron 0 firing-rate result:")
print(fr_neuron0)

YFF neuron 0 firing-rate result:
{'neuron': 0, 'fr_accuracy': 0.09174311926605505, 'permutation_p': 0.7711442786069652, 'n_perm_equal_or_better': 154, 'coding': False, 'valid': True, 'invalid_reason': ''}


In [110]:
X_fr0, y_fr0 = make_pooled_temporal_features(
    spikes=spikes_yff,
    pooled_table=pooled_yff,
    neuron_idx=0,
    bin_ms=900
)

fr_direct_accuracy = matlab_gamma_lda_cv(
    X_fr0,
    y_fr0,
    gamma=0.2,
    random_state=0
)

print("\nDirect 900-ms accuracy:")
print(fr_direct_accuracy)

print("\nFR-function accuracy:")
print(fr_neuron0["fr_accuracy"])

print("\nDifference:")
print(
    fr_neuron0["fr_accuracy"]
    - fr_direct_accuracy
)


Direct 900-ms accuracy:
0.09174311926605505

FR-function accuracy:
0.09174311926605505

Difference:
0.0


In [111]:
# ============================================================
# YFF — FIRING-RATE CODING ANALYSIS
# 37 MTL neurons, 200 permutations each
# ============================================================

yff_fr_results = []

for counter, (_, neuron_row) in enumerate(
    yff_mtl.iterrows(),
    start=1
):

    neuron_idx = int(
        neuron_row["neuron"]
    )

    print(
        f"YFF FR: "
        f"{counter}/{len(yff_mtl)} "
        f"neuron={neuron_idx} "
        f"region={neuron_row['region_raw']}"
    )

    result = analyze_one_firing_rate_neuron(
        spikes=spikes_yff,
        pooled_table=pooled_yff,
        neuron_idx=neuron_idx,
        n_permutations=200,
        random_state=0
    )

    result["subject"] = "YFF"
    result["region_raw"] = neuron_row["region_raw"]
    result["region"] = neuron_row["region"]

    yff_fr_results.append(
        result
    )


yff_fr = pd.DataFrame(
    yff_fr_results
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("YFF FIRING-RATE RESULTS")
print("=" * 60)

print(
    "Total MTL neurons:",
    len(yff_fr)
)

print(
    "Valid neurons:",
    yff_fr["valid"].sum()
)

print(
    "Invalid neurons:",
    (~yff_fr["valid"]).sum()
)

print(
    "FR coding neurons:",
    yff_fr["coding"].sum()
)

print(
    "FR coding percentage — all neurons:",
    100
    * yff_fr["coding"].sum()
    / len(yff_fr)
)

if yff_fr["valid"].sum() > 0:

    print(
        "FR coding percentage — valid neurons:",
        100
        * yff_fr["coding"].sum()
        / yff_fr["valid"].sum()
    )


print("\nInvalid reasons:")

print(
    yff_fr.loc[
        ~yff_fr["valid"],
        "invalid_reason"
    ].value_counts()
)


display(
    yff_fr[
        [
            "neuron",
            "region_raw",
            "fr_accuracy",
            "permutation_p",
            "coding",
            "valid",
            "invalid_reason"
        ]
    ]
)

YFF FR: 1/37 neuron=8 region=ent
YFF FR: 2/37 neuron=9 region=hpc
YFF FR: 3/37 neuron=10 region=hpc
YFF FR: 4/37 neuron=11 region=hpc
YFF FR: 5/37 neuron=12 region=hpc
YFF FR: 6/37 neuron=13 region=hpc
YFF FR: 7/37 neuron=14 region=hpc
YFF FR: 8/37 neuron=15 region=hpc
YFF FR: 9/37 neuron=16 region=hpc
YFF FR: 10/37 neuron=26 region=hpc
YFF FR: 11/37 neuron=27 region=hpc
YFF FR: 12/37 neuron=28 region=hpc
YFF FR: 13/37 neuron=29 region=hpc
YFF FR: 14/37 neuron=30 region=hpc
YFF FR: 15/37 neuron=31 region=hpc
YFF FR: 16/37 neuron=32 region=hpc
YFF FR: 17/37 neuron=33 region=hpc
YFF FR: 18/37 neuron=34 region=hpc
YFF FR: 19/37 neuron=35 region=hpc
YFF FR: 20/37 neuron=36 region=hpc
YFF FR: 21/37 neuron=37 region=hpc
YFF FR: 22/37 neuron=38 region=hpc
YFF FR: 23/37 neuron=39 region=hpc
YFF FR: 24/37 neuron=40 region=hpc
YFF FR: 25/37 neuron=41 region=hpc
YFF FR: 26/37 neuron=42 region=hpc
YFF FR: 27/37 neuron=43 region=hpc
YFF FR: 28/37 neuron=44 region=hpc
YFF FR: 29/37 neuron=45 region=

,neuron,region_raw,fr_accuracy,permutation_p,coding,valid,invalid_reason
0,8,ent,0.100917,0.701493,False,True,
1,9,hpc,0.128440,0.333333,False,True,
2,10,hpc,0.119266,0.368159,False,True,
3,11,hpc,0.128440,0.338308,False,True,
4,12,hpc,0.174312,0.034826,True,True,
5,13,hpc,0.128440,0.318408,False,True,
6,14,hpc,0.064220,0.915423,False,True,
7,15,hpc,0.091743,0.636816,False,True,
8,16,hpc,0.165138,0.074627,False,True,
9,26,hpc,0.055046,0.975124,False,True,


In [112]:
# ============================================================
# ALL 11 SUBJECTS — FIRING-RATE CODING
# 554 MTL neurons
# ============================================================

all_fr_results = []

for subject in SUBJECTS:

    print("\n" + "=" * 65)
    print(f"SUBJECT: {subject}")
    print("=" * 65)

    # --------------------------------------------------------
    # Load spikes
    # --------------------------------------------------------

    spike_path = find_arithmetic_spike_file(
        subject
    )

    spikes_subj = load_arithmetic_spikes(
        spike_path
    )

    # --------------------------------------------------------
    # Prepare behavior
    # --------------------------------------------------------

    behav_subj, pooled_subj = (
        prepare_arithmetic_subject(
            subject,
            spikes_subj
        )
    )

    # --------------------------------------------------------
    # MTL neurons for this subject
    # --------------------------------------------------------

    subject_mtl = (
        mtl_neurons.loc[
            mtl_neurons["subject"] == subject
        ]
        .copy()
        .sort_values("neuron")
    )

    n_subject_mtl = len(
        subject_mtl
    )

    print(
        f"MTL neurons: {n_subject_mtl}"
    )

    # --------------------------------------------------------
    # Analyze every MTL neuron
    # --------------------------------------------------------

    for counter, (_, neuron_row) in enumerate(
        subject_mtl.iterrows(),
        start=1
    ):

        neuron_idx = int(
            neuron_row["neuron"]
        )

        print(
            f"{subject}: "
            f"{counter}/{n_subject_mtl} "
            f"neuron={neuron_idx} "
            f"region={neuron_row['region_raw']}"
        )

        result = analyze_one_firing_rate_neuron(
            spikes=spikes_subj,
            pooled_table=pooled_subj,
            neuron_idx=neuron_idx,
            n_permutations=200,
            random_state=0
        )

        result["subject"] = subject
        result["region_raw"] = (
            neuron_row["region_raw"]
        )
        result["region"] = (
            neuron_row["region"]
        )

        all_fr_results.append(
            result
        )


# ============================================================
# BUILD NEURON-LEVEL TABLE
# ============================================================

fr_mtl = pd.DataFrame(
    all_fr_results
)

column_order = [
    "subject",
    "neuron",
    "region_raw",
    "region",
    "fr_accuracy",
    "permutation_p",
    "n_perm_equal_or_better",
    "coding",
    "valid",
    "invalid_reason"
]

fr_mtl = fr_mtl[
    column_order
]


# ============================================================
# SUBJECT-LEVEL SUMMARY
# ============================================================

fr_subject_summary = (
    fr_mtl
    .groupby("subject")
    .agg(
        total_mtl_neurons=("neuron", "count"),
        valid_neurons=("valid", "sum"),
        coding_neurons=("coding", "sum")
    )
    .reset_index()
)

fr_subject_summary[
    "invalid_neurons"
] = (
    fr_subject_summary[
        "total_mtl_neurons"
    ]
    -
    fr_subject_summary[
        "valid_neurons"
    ]
)

# Use all MTL neurons as denominator
fr_subject_summary[
    "coding_percentage"
] = (
    100
    * fr_subject_summary[
        "coding_neurons"
    ]
    /
    fr_subject_summary[
        "total_mtl_neurons"
    ]
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 65)
print("ALL-SUBJECT FIRING-RATE RESULTS")
print("=" * 65)

print(
    "Total MTL neurons:",
    len(fr_mtl)
)

print(
    "Valid neurons:",
    fr_mtl["valid"].sum()
)

print(
    "Invalid neurons:",
    (~fr_mtl["valid"]).sum()
)

print(
    "FR coding neurons:",
    fr_mtl["coding"].sum()
)

print(
    "\nPooled FR coding percentage:",
    100
    * fr_mtl["coding"].sum()
    / len(fr_mtl)
)

print(
    "\nMean subject-level FR coding percentage:",
    fr_subject_summary[
        "coding_percentage"
    ].mean()
)

print(
    "SEM across subjects:",
    fr_subject_summary[
        "coding_percentage"
    ].sem()
)

print("\nSubject-level results:")

display(
    fr_subject_summary
)


SUBJECT: YFF
MTL neurons: 37
YFF: 1/37 neuron=8 region=ent
YFF: 2/37 neuron=9 region=hpc
YFF: 3/37 neuron=10 region=hpc
YFF: 4/37 neuron=11 region=hpc
YFF: 5/37 neuron=12 region=hpc
YFF: 6/37 neuron=13 region=hpc
YFF: 7/37 neuron=14 region=hpc
YFF: 8/37 neuron=15 region=hpc
YFF: 9/37 neuron=16 region=hpc
YFF: 10/37 neuron=26 region=hpc
YFF: 11/37 neuron=27 region=hpc
YFF: 12/37 neuron=28 region=hpc
YFF: 13/37 neuron=29 region=hpc
YFF: 14/37 neuron=30 region=hpc
YFF: 15/37 neuron=31 region=hpc
YFF: 16/37 neuron=32 region=hpc
YFF: 17/37 neuron=33 region=hpc
YFF: 18/37 neuron=34 region=hpc
YFF: 19/37 neuron=35 region=hpc
YFF: 20/37 neuron=36 region=hpc
YFF: 21/37 neuron=37 region=hpc
YFF: 22/37 neuron=38 region=hpc
YFF: 23/37 neuron=39 region=hpc
YFF: 24/37 neuron=40 region=hpc
YFF: 25/37 neuron=41 region=hpc
YFF: 26/37 neuron=42 region=hpc
YFF: 27/37 neuron=43 region=hpc
YFF: 28/37 neuron=44 region=hpc
YFF: 29/37 neuron=45 region=hpc
YFF: 30/37 neuron=46 region=hpc
YFF: 31/37 neuron=47 

,subject,total_mtl_neurons,valid_neurons,coding_neurons,invalid_neurons,coding_percentage
0,YFF,37,37,1,0,2.702703
1,YFI,29,29,2,0,6.896552
2,YFJ,45,45,2,0,4.444444
3,YFK,44,42,1,2,2.272727
4,YFL,57,56,1,1,1.754386
5,YFM,61,58,1,3,1.639344
6,YFP,43,41,3,2,6.976744
7,YFR,65,64,2,1,3.076923
8,YFS,59,58,2,1,3.389831
9,YFT,52,52,5,0,9.615385


In [113]:
fr_mtl.to_csv(
    TABLES / "arithmetic_firing_rate_mtl_neuron_results.csv",
    index=False
)

fr_subject_summary.to_csv(
    TABLES / "arithmetic_firing_rate_subject_summary.csv",
    index=False
)

print("FR tables saved.")

FR tables saved.


In [114]:
print("=" * 65)
print("FINAL FIRING-RATE REPRODUCTION SUMMARY")
print("=" * 65)

print("Total MTL neurons:", len(fr_mtl))
print("Valid neurons:", fr_mtl["valid"].sum())
print("Invalid neurons:", (~fr_mtl["valid"]).sum())
print("FR coding neurons:", fr_mtl["coding"].sum())

print(
    "\nPooled FR coding percentage:",
    100 * fr_mtl["coding"].sum() / len(fr_mtl)
)

print(
    "\nMean subject-level FR coding percentage:",
    fr_subject_summary["coding_percentage"].mean()
)

print(
    "SEM across subjects:",
    fr_subject_summary["coding_percentage"].sem()
)

print("\nSubject-level results:")
display(fr_subject_summary)

FINAL FIRING-RATE REPRODUCTION SUMMARY
Total MTL neurons: 554
Valid neurons: 544
Invalid neurons: 10
FR coding neurons: 25

Pooled FR coding percentage: 4.512635379061372

Mean subject-level FR coding percentage: 4.621232262461887
SEM across subjects: 0.8407921954673638

Subject-level results:


,subject,total_mtl_neurons,valid_neurons,coding_neurons,invalid_neurons,coding_percentage
0,YFF,37,37,1,0,2.702703
1,YFI,29,29,2,0,6.896552
2,YFJ,45,45,2,0,4.444444
3,YFK,44,42,1,2,2.272727
4,YFL,57,56,1,1,1.754386
5,YFM,61,58,1,3,1.639344
6,YFP,43,41,3,2,6.976744
7,YFR,65,64,2,1,3.076923
8,YFS,59,58,2,1,3.389831
9,YFT,52,52,5,0,9.615385


In [115]:
# ============================================================
# YFF FIRING-RATE CODING ACROSS CV SEEDS
# ============================================================

FR_SEEDS = [0, 1, 2, 3, 4]

yff_fr_seed_results = []

for seed in FR_SEEDS:

    print("\n" + "=" * 60)
    print(f"YFF FR — SEED {seed}")
    print("=" * 60)

    for counter, (_, neuron_row) in enumerate(
        yff_mtl.iterrows(),
        start=1
    ):

        neuron_idx = int(
            neuron_row["neuron"]
        )

        result = analyze_one_firing_rate_neuron(
            spikes=spikes_yff,
            pooled_table=pooled_yff,
            neuron_idx=neuron_idx,
            n_permutations=200,
            random_state=seed
        )

        result["seed"] = seed
        result["subject"] = "YFF"
        result["region_raw"] = neuron_row["region_raw"]

        yff_fr_seed_results.append(
            result
        )

        print(
            f"{counter}/{len(yff_mtl)} "
            f"neuron={neuron_idx} "
            f"coding={result['coding']} "
            f"p={result['permutation_p']}"
        )


yff_fr_seed = pd.DataFrame(
    yff_fr_seed_results
)


# ============================================================
# SUMMARY
# ============================================================

yff_fr_seed_summary = (
    yff_fr_seed
    .groupby("seed")
    .agg(
        total_neurons=("neuron", "count"),
        valid_neurons=("valid", "sum"),
        coding_neurons=("coding", "sum")
    )
    .reset_index()
)

yff_fr_seed_summary["coding_percentage"] = (
    100
    * yff_fr_seed_summary["coding_neurons"]
    / yff_fr_seed_summary["total_neurons"]
)

display(yff_fr_seed_summary)

print("\nCoding-neuron range:")
print(
    yff_fr_seed_summary["coding_neurons"].min(),
    "to",
    yff_fr_seed_summary["coding_neurons"].max()
)

print("\nCoding-percentage range:")
print(
    yff_fr_seed_summary["coding_percentage"].min(),
    "to",
    yff_fr_seed_summary["coding_percentage"].max()
)


YFF FR — SEED 0
1/37 neuron=8 coding=False p=0.7014925373134329
2/37 neuron=9 coding=False p=0.3333333333333333
3/37 neuron=10 coding=False p=0.3681592039800995
4/37 neuron=11 coding=False p=0.3383084577114428
5/37 neuron=12 coding=True p=0.03482587064676617
6/37 neuron=13 coding=False p=0.31840796019900497
7/37 neuron=14 coding=False p=0.9154228855721394
8/37 neuron=15 coding=False p=0.6368159203980099
9/37 neuron=16 coding=False p=0.07462686567164178
10/37 neuron=26 coding=False p=0.9751243781094527
11/37 neuron=27 coding=False p=0.9253731343283582
12/37 neuron=28 coding=False p=0.9601990049751243
13/37 neuron=29 coding=False p=0.9502487562189055
14/37 neuron=30 coding=False p=0.8855721393034826
15/37 neuron=31 coding=False p=0.9502487562189055
16/37 neuron=32 coding=False p=0.2537313432835821
17/37 neuron=33 coding=False p=0.5323383084577115
18/37 neuron=34 coding=False p=0.6268656716417911
19/37 neuron=35 coding=False p=0.6318407960199005
20/37 neuron=36 coding=False p=1.0
21/37 n

,seed,total_neurons,valid_neurons,coding_neurons,coding_percentage
0,0,37,37,1,2.702703
1,1,37,37,1,2.702703
2,2,37,37,1,2.702703
3,3,37,37,2,5.405405
4,4,37,37,2,5.405405



Coding-neuron range:
1 to 2

Coding-percentage range:
2.7027027027027026 to 5.405405405405405


Paired T test 

In [116]:
from scipy.stats import ttest_rel

# ============================================================
# MERGE TEMPORAL AND FIRING-RATE SUBJECT RESULTS
# ============================================================

# Temporal percentage using ALL MTL neurons as denominator
temporal_for_test = (
    temporal_mtl
    .groupby("subject")
    .agg(
        total_mtl_neurons=("neuron", "count"),
        temporal_coding_neurons=("coding", "sum")
    )
    .reset_index()
)

temporal_for_test["temporal_percentage"] = (
    100
    * temporal_for_test["temporal_coding_neurons"]
    / temporal_for_test["total_mtl_neurons"]
)


# Firing-rate percentage
fr_for_test = (
    fr_mtl
    .groupby("subject")
    .agg(
        fr_coding_neurons=("coding", "sum")
    )
    .reset_index()
)


comparison = temporal_for_test.merge(
    fr_for_test,
    on="subject"
)

comparison["fr_percentage"] = (
    100
    * comparison["fr_coding_neurons"]
    / comparison["total_mtl_neurons"]
)

comparison["difference_percentage_points"] = (
    comparison["temporal_percentage"]
    -
    comparison["fr_percentage"]
)


# ============================================================
# PAIRED T-TEST
# ============================================================

t_stat, p_value = ttest_rel(
    comparison["temporal_percentage"],
    comparison["fr_percentage"]
)


print("=" * 65)
print("TEMPORAL vs FIRING-RATE — PAIRED SUBJECT TEST")
print("=" * 65)

display(comparison)

print(
    "\nMean temporal percentage:",
    comparison["temporal_percentage"].mean()
)

print(
    "Temporal SEM:",
    comparison["temporal_percentage"].sem()
)

print(
    "\nMean FR percentage:",
    comparison["fr_percentage"].mean()
)

print(
    "FR SEM:",
    comparison["fr_percentage"].sem()
)

print(
    "\nMean temporal - FR difference:",
    comparison["difference_percentage_points"].mean()
)

print("\nPaired t statistic:")
print(t_stat)

print("\nPaired t-test p-value:")
print(p_value)

TEMPORAL vs FIRING-RATE — PAIRED SUBJECT TEST


,subject,total_mtl_neurons,temporal_coding_neurons,temporal_percentage,fr_coding_neurons,fr_percentage,difference_percentage_points
0,YFF,37,11,29.729730,1,2.702703,27.027027
1,YFI,29,8,27.586207,2,6.896552,20.689655
2,YFJ,45,13,28.888889,2,4.444444,24.444444
3,YFK,44,11,25.000000,1,2.272727,22.727273
4,YFL,57,13,22.807018,1,1.754386,21.052632
5,YFM,61,17,27.868852,1,1.639344,26.229508
6,YFP,43,10,23.255814,3,6.976744,16.279070
7,YFR,65,17,26.153846,2,3.076923,23.076923
8,YFS,59,21,35.593220,2,3.389831,32.203390
9,YFT,52,16,30.769231,5,9.615385,21.153846



Mean temporal percentage: 28.61359826610392
Temporal SEM: 1.3810028260329825

Mean FR percentage: 4.621232262461887
FR SEM: 0.8407921954673638

Mean temporal - FR difference: 23.992366003642033

Paired t statistic:
17.938187843985826

Paired t-test p-value:
6.199914558304843e-09


So conceptually:

$$ \boxed{\text{Temporal representation} \gg \text{Firing-rate representation}} $$

is reproduced very clearly.

In [117]:
comparison.to_csv(
    TABLES / "arithmetic_temporal_vs_firing_rate_subject_comparison.csv",
    index=False
)

print(
    "Saved:",
    TABLES / "arithmetic_temporal_vs_firing_rate_subject_comparison.csv"
)

Saved: C:\Users\shafi\number-simplex-reproduction\tables\arithmetic_temporal_vs_firing_rate_subject_comparison.csv


Paper: temporal \(24.80\pm1.73\%\), FR \(3.36\pm0.71\%\).
Reproduction: temporal \(28.61\pm1.38\%\), FR \(4.62\pm0.84\%\).
Reproduction still yields a very strong paired temporal-vs-FR difference, \(t(10)=17.94,\ p=6.20\times10^{-9}\). Exact coding proportions are sensitive to the unspecified CV partition; for YFF, temporal coding varied from 24.32% to 32.43% across five CV seeds

# regional specificity test

binomial GLMM

IsTuned∼Region+(1∣Subject)

neuron    region       coding
--------------------------------
12        hpc          1
13        hpc          0
14        amy          1
15        ent          0
...

In [118]:
# ============================================================
# TEMPORAL CODING BY MTL REGION
# DESCRIPTIVE CHECK BEFORE STATISTICAL MODEL
# ============================================================

region_temporal_summary = (
    temporal_mtl
    .groupby(
        ["region_raw", "region"]
    )
    .agg(
        total_neurons=("neuron", "count"),
        valid_neurons=("valid", "sum"),
        coding_neurons=("coding", "sum")
    )
    .reset_index()
)


region_temporal_summary[
    "invalid_neurons"
] = (
    region_temporal_summary[
        "total_neurons"
    ]
    -
    region_temporal_summary[
        "valid_neurons"
    ]
)


region_temporal_summary[
    "coding_percentage"
] = (
    100
    * region_temporal_summary[
        "coding_neurons"
    ]
    /
    region_temporal_summary[
        "total_neurons"
    ]
)


# Put regions in a sensible order
region_order = [
    "hpc",
    "ent",
    "amy",
    "para-hpc"
]

region_temporal_summary[
    "region_raw"
] = pd.Categorical(
    region_temporal_summary[
        "region_raw"
    ],
    categories=region_order,
    ordered=True
)

region_temporal_summary = (
    region_temporal_summary
    .sort_values("region_raw")
    .reset_index(drop=True)
)


display(
    region_temporal_summary
)


print("\nTotal neurons:")
print(
    region_temporal_summary[
        "total_neurons"
    ].sum()
)

print("\nTotal coding neurons:")
print(
    region_temporal_summary[
        "coding_neurons"
    ].sum()
)


,region_raw,region,total_neurons,valid_neurons,coding_neurons,invalid_neurons,coding_percentage
0,hpc,HPC,389,379,113,10,29.048843
1,ent,ENT,70,70,19,0,27.142857
2,amy,AMY,77,77,23,0,29.870130
3,para-hpc,PARA-HPC,18,18,5,0,27.777778



Total neurons:
554

Total coding neurons:
160


In [119]:
# ============================================================
# STEP 3
# HOW ARE MTL NEURONS DISTRIBUTED ACROSS SUBJECTS AND REGIONS?
# ============================================================

subject_region_counts = pd.crosstab(
    temporal_mtl["subject"],
    temporal_mtl["region_raw"]
)

# Put columns in paper/dataset order
subject_region_counts = subject_region_counts.reindex(
    columns=["hpc", "ent", "amy", "para-hpc"],
    fill_value=0
)

subject_region_counts["TOTAL"] = (
    subject_region_counts.sum(axis=1)
)

display(subject_region_counts)


print("\nNumber of subjects contributing to each region:")

for region in ["hpc", "ent", "amy", "para-hpc"]:
    n_subjects = (
        subject_region_counts[region] > 0
    ).sum()

    print(
        f"{region:10s}: {n_subjects} subjects"
    )


print("\nTotal neurons by region:")

print(
    subject_region_counts[
        ["hpc", "ent", "amy", "para-hpc"]
    ].sum()
)

region_raw,hpc,ent,amy,para-hpc,TOTAL
subject,,,,,
YFF,36,1,0,0,37
YFI,21,8,0,0,29
YFJ,45,0,0,0,45
YFK,26,0,0,18,44
YFL,42,15,0,0,57
YFM,32,29,0,0,61
YFP,43,0,0,0,43
YFR,41,0,24,0,65
YFS,37,0,22,0,59



Number of subjects contributing to each region:
hpc       : 11 subjects
ent       : 5 subjects
amy       : 4 subjects
para-hpc  : 1 subjects

Total neurons by region:
region_raw
hpc         389
ent          70
amy          77
para-hpc     18
dtype: int64


In [120]:
# ============================================================
# STEP 4
# BINOMIAL MIXED MODEL:
# CODING ~ REGION + (1 | SUBJECT)
# ============================================================

import statsmodels.api as sm
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM


# ------------------------------------------------------------
# Build clean model dataframe
# ------------------------------------------------------------

glmm_data = temporal_mtl[
    [
        "subject",
        "region_raw",
        "coding"
    ]
].copy()


# coding must be numerical 0/1
glmm_data["coding"] = (
    glmm_data["coding"]
    .astype(int)
)


# Explicit region order:
# HPC becomes the reference category
glmm_data["region_raw"] = pd.Categorical(
    glmm_data["region_raw"],
    categories=[
        "hpc",
        "ent",
        "amy",
        "para-hpc"
    ],
    ordered=True
)


print("Number of neurons:", len(glmm_data))

print("\nCoding counts:")
print(
    glmm_data["coding"].value_counts()
)

print("\nRegion counts:")
print(
    glmm_data["region_raw"].value_counts(
        sort=False
    )
)

print("\nSubject count:")
print(
    glmm_data["subject"].nunique()
)


# ------------------------------------------------------------
# Model
#
# Fixed effect:
#     region
#
# Random effect:
#     subject-specific intercept
# ------------------------------------------------------------

random_effects = {
    "Subject": "0 + C(subject)"
}


glmm_model = BinomialBayesMixedGLM.from_formula(
    "coding ~ C(region_raw, Treatment(reference='hpc'))",
    random_effects,
    glmm_data
)


# Variational-Bayes fit
glmm_result = glmm_model.fit_vb()


print("\n" + "=" * 70)
print("BINOMIAL MIXED MODEL")
print("coding ~ region + (1 | subject)")
print("=" * 70)

print(
    glmm_result.summary()
)

Number of neurons: 554

Coding counts:
coding
0    394
1    160
Name: count, dtype: int64

Region counts:
region_raw
hpc         389
ent          70
amy          77
para-hpc     18
Name: count, dtype: int64

Subject count:
11

BINOMIAL MIXED MODEL
coding ~ region + (1 | subject)
                                     Binomial Mixed GLM Results
                                                      Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
----------------------------------------------------------------------------------------------------
Intercept                                                M    -0.8957   0.0937                      
C(region_raw, Treatment(reference='hpc'))[T.ent]         M    -0.0989   0.2676                      
C(region_raw, Treatment(reference='hpc'))[T.amy]         M    -0.0265   0.2482                      
C(region_raw, Treatment(reference='hpc'))[T.para-hpc]    M    -0.0732   0.5196                      
Subject                                           

In [122]:
glmm_result.summary()

,Type,Post. Mean,Post. SD,SD,SD (LB),SD (UB)
Intercept,M,-0.8957,0.0937,,,
"C(region_raw, Treatment(reference='hpc'))[T.ent]",M,-0.0989,0.2676,,,
"C(region_raw, Treatment(reference='hpc'))[T.amy]",M,-0.0265,0.2482,,,
"C(region_raw, Treatment(reference='hpc'))[T.para-hpc]",M,-0.0732,0.5196,,,
Subject,V,-1.6510,0.2253,0.192,0.122,0.301


HPC
ENT
AMY
PARA-HPC
	​

:29.05%
:27.14%
:29.87%
:27.78%.
	​


In [123]:
# ============================================================
# STEP 5
# JOINT TEST OF THE THREE REGION COEFFICIENTS
# ============================================================

from scipy.stats import chi2
import numpy as np


# Fixed-effect posterior means
fixed_means = np.asarray(
    glmm_result.fe_mean
)


# Fixed-effect posterior SDs
fixed_sds = np.asarray(
    glmm_result.fe_sd
)


print("=" * 70)
print("FIXED EFFECTS")
print("=" * 70)

for name, mean, sd in zip(
    glmm_model.exog_names,
    fixed_means,
    fixed_sds
):
    print(
        f"{name}\n"
        f"    estimate = {mean:.6f}\n"
        f"    SD       = {sd:.6f}\n"
    )


# ------------------------------------------------------------
# Region coefficients are positions 1, 2, 3
# Position 0 is the intercept (HPC reference)
# ------------------------------------------------------------

region_beta = fixed_means[1:4]

region_var = fixed_sds[1:4] ** 2


# Approximate joint Wald statistic
#
# Sum of squared standardized regional coefficients
#
wald_chi2 = np.sum(
    (region_beta ** 2) / region_var
)

df = 3

wald_p = chi2.sf(
    wald_chi2,
    df=df
)


print("=" * 70)
print("APPROXIMATE JOINT REGION TEST")
print("=" * 70)

print(
    "Wald chi-square:",
    wald_chi2
)

print(
    "Degrees of freedom:",
    df
)

print(
    "p-value:",
    wald_p
)

FIXED EFFECTS
Intercept
    estimate = -0.895711
    SD       = 0.093716

C(region_raw, Treatment(reference='hpc'))[T.ent]
    estimate = -0.098852
    SD       = 0.267606

C(region_raw, Treatment(reference='hpc'))[T.amy]
    estimate = -0.026476
    SD       = 0.248234

C(region_raw, Treatment(reference='hpc'))[T.para-hpc]
    estimate = -0.073247
    SD       = 0.519618

APPROXIMATE JOINT REGION TEST
Wald chi-square: 0.16769789653818548
Degrees of freedom: 3
p-value: 0.9826273491543661


In [124]:
# ============================================================
# STEP 6
# EXPORT NEURON-LEVEL DATA FOR EXACT MATLAB GLMM
# ============================================================

region_glmm_export = temporal_mtl[
    [
        "subject",
        "region_raw",
        "neuron",
        "coding"
    ]
].copy()


# Convert coding to integer 0/1
region_glmm_export["coding"] = (
    region_glmm_export["coding"]
    .astype(int)
)


# Rename columns clearly for MATLAB
region_glmm_export = (
    region_glmm_export.rename(
        columns={
            "subject": "Subject",
            "region_raw": "Region",
            "neuron": "Neuron",
            "coding": "IsTuned"
        }
    )
)


# Save
glmm_export_path = (
    TABLES
    / "arithmetic_temporal_region_glmm_input.csv"
)

region_glmm_export.to_csv(
    glmm_export_path,
    index=False
)


print("=" * 65)
print("MATLAB GLMM INPUT")
print("=" * 65)

print("Saved to:")
print(glmm_export_path)

print("\nShape:")
print(region_glmm_export.shape)

print("\nCoding counts:")
print(
    region_glmm_export[
        "IsTuned"
    ].value_counts()
)

print("\nRegion counts:")
print(
    region_glmm_export[
        "Region"
    ].value_counts()
)

print("\nSubjects:")
print(
    region_glmm_export[
        "Subject"
    ].nunique()
)

display(
    region_glmm_export.head(10)
)

MATLAB GLMM INPUT
Saved to:
C:\Users\shafi\number-simplex-reproduction\tables\arithmetic_temporal_region_glmm_input.csv

Shape:
(554, 4)

Coding counts:
IsTuned
0    394
1    160
Name: count, dtype: int64

Region counts:
Region
hpc         389
amy          77
ent          70
para-hpc     18
Name: count, dtype: int64

Subjects:
11


,Subject,Region,Neuron,IsTuned
0,YFF,ent,8,0
1,YFF,hpc,9,1
2,YFF,hpc,10,0
3,YFF,hpc,11,0
4,YFF,hpc,12,1
5,YFF,hpc,13,1
6,YFF,hpc,14,0
7,YFF,hpc,15,0
8,YFF,hpc,16,0
9,YFF,hpc,26,0
